In [1]:
import sqlite3
import pandas as pd

key_cols = ["V010100", "NUM_QUADRA", "NUM_FACE", "V010800"]

Validando e buscando exemplos para as diretrizes de anotação do slide: https://docs.google.com/presentation/d/1je9s2kncPsZYMCoezqWOdqXF9XKOHHcq6R_EEuEADvY/edit?usp=sharing

In [2]:
DB_CRIT = "../data/amstr_geral_criticas.db"
DB_MAIN = DB_CRIT

def compilar_regra(df_final: pd.DataFrame, query: str, titulo_regra: str,
                      db_path: str = DB_CRIT):
    with sqlite3.connect(db_path) as conn:
        df_regra = pd.read_sql_query(query, conn)

    if not df_regra.empty:
        df_regra = df_regra[key_cols].drop_duplicates().copy()
        df_regra[titulo_regra] = 1
    else:
        df_regra = pd.DataFrame(columns=[*key_cols, titulo_regra])

    if df_final.empty:
        df_final = df_regra
    else:
        if titulo_regra in df_final.columns:
            df_final = df_final.drop(columns=[titulo_regra])
        df_final = pd.merge(df_final, df_regra, on=key_cols, how='left').fillna({titulo_regra: 0})

    if titulo_regra not in df_final.columns:
        df_final[titulo_regra] = 0

    df_final[titulo_regra] = df_final[titulo_regra].astype('Int8')
    return df_final

def fetch_keys(query: str, db_path: str) -> pd.DataFrame:
    with sqlite3.connect(db_path) as conn:
        df = pd.read_sql_query(query, conn)
    if df.empty:
        return pd.DataFrame(columns=key_cols)
    return df[key_cols].drop_duplicates().reset_index(drop=True)


def aplicar_regra_sql(nome_regra: str, query: str, db_path: str = DB_CRIT):
    global df_estabelecimentos
    if query is None or str(query).strip() == "":
        aplicar_regra_vazia(nome_regra)
        return
    df_estabelecimentos = compilar_regra(df_estabelecimentos, query, nome_regra, db_path=db_path)

def aplicar_regra_df(nome_regra: str, df_keys: pd.DataFrame):
    global df_estabelecimentos

    if df_keys is None or df_keys.empty:
        df_regra = pd.DataFrame(columns=[*key_cols, nome_regra])
    else:
        df_regra = df_keys[key_cols].drop_duplicates().copy()
        df_regra[nome_regra] = 1

    if nome_regra in df_estabelecimentos.columns:
        df_estabelecimentos = df_estabelecimentos.drop(columns=[nome_regra])

    df_estabelecimentos = pd.merge(df_estabelecimentos, df_regra, on=key_cols, how="left")
    if nome_regra not in df_estabelecimentos.columns:
        df_estabelecimentos[nome_regra] = 0

    df_estabelecimentos[nome_regra] = pd.to_numeric(
        df_estabelecimentos[nome_regra], errors="coerce"
    ).fillna(0).astype("Int8")

def aplicar_regra_vazia(nome_regra: str):
    aplicar_regra_df(nome_regra, pd.DataFrame(columns=key_cols))

def preparar_base_estabelecimentos():
    with sqlite3.connect(DB_CRIT) as conn:
        df_estabelecimentos = pd.read_sql_query(
            f"SELECT {', '.join(key_cols)} FROM estabel",
            conn
        )
    return df_estabelecimentos[key_cols].drop_duplicates().reset_index(drop=True)

In [3]:
with sqlite3.connect(DB_CRIT) as conn:
    cursor = conn.cursor()
    cursor.execute(f""" SELECT count(*) FROM estabel""")
    QTD_ESTABELECIMENTOS = cursor.fetchone()[0]

print(f"Total de estabelecimentos: {QTD_ESTABELECIMENTOS}")

Total de estabelecimentos: 100000


# Não Anomalia

## Regra 1

In [4]:
query = """
SELECT
    COUNT(*) as total
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica = 6
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 1714 -> 1.71%


## Regra 2

In [5]:
query = """
SELECT
    COUNT(*) as total
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica = 382
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 54088 -> 54.09%


# Anomalia

## Regra 1

In [6]:
#Sem SQL para essa regra, depende de avaliação manual
query_r1 = f""" SELECT {', '.join(key_cols)} FROM estabel"""
# df_estabelecimentos = compilador_regras(pd.DataFrame(), query_r1, "R01")

## Regra 2

In [7]:
orientacao_tecnica = "V05120200 = 2"
escolaridade = "V02220000 IN ('01', '02', '03', '04', '05', '06', '07')"

query_r2_1 = f"""
    SELECT count(*)
    FROM estabel
    WHERE {orientacao_tecnica} AND {escolaridade}
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query_r2_1)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 4917 -> 4.92%


Considerando os que SOMENTE recebem orientação própria, mesmo que não sejam qualificados para isso

In [8]:
orientacao_tecnica = "(V05120200 = 2 AND V05120100 != 2 AND V05120300 != 2 AND V05120400 != 2 AND V05120500 != 2 AND V05120600 != 2 AND V05120700 != 2 AND V05120800 != 2)"
escolaridade = "V02220000 IN ('01', '02', '03', '04', '05', '06', '07')"

query_r2_2 = f"""
    SELECT COUNT(*)
    FROM estabel
    WHERE {orientacao_tecnica} AND {escolaridade}
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query_r2_2)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 3752 -> 3.75%


In [9]:
query_r2 = """
SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800
FROM estabel
WHERE
    (V05120200 = 2
    AND V05120100 != 2 AND V05120300 != 2 AND V05120400 != 2
    AND V05120500 != 2 AND V05120600 != 2 AND V05120700 != 2 AND V05120800 != 2)
    AND V02220000 IN ('01', '02', '03', '04', '05', '06', '07')
"""

## Regra 3

### Caso 1 - Lavoura Permanente

In [10]:
ausencia_lav_per = "(e.VW04020000 IS NULL OR e.VW04020000 = 0)"
join_lav_per = " AND ".join([f"e.{col} = lp.{col}" for col in key_cols])

query = f"""
    SELECT COUNT(*)
    FROM (
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN lav_per lp
            ON {join_lav_per}
        WHERE {ausencia_lav_per}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    ) estabels_com_lav_per
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 3206 -> 3.21%


### Caso 2 - Lavoura Temporaria

In [11]:
ausencia_lav_temp = "(e.VW04030000 IS NULL OR e.VW04030000 = 0)"
join_lav_temp = " AND ".join([f"e.{col} = lt.{col}" for col in key_cols])

query = f"""
    SELECT COUNT(*)
    FROM (
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN lav_temp lt
            ON {join_lav_temp}
        WHERE {ausencia_lav_temp}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    ) estabels_com_lav_temp
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 2994 -> 2.99%


### Caso 3 - Horticultura

In [12]:
ausencia_hort = "(e.VW04030000 IS NULL OR e.VW04030000 = 0)"
join_hort = " AND ".join([f"e.{col} = ht.{col}" for col in key_cols])

# Verificando quantos estabelecimentos têm registros na tabela hort, mas não possuem informações de area
query = f"""
    SELECT COUNT(*)
    FROM (
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN hort ht
            ON {join_hort}
        WHERE {ausencia_hort}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    ) estabels_com_hort
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 274 -> 0.27%


### Caso 4 - Silvicultura

In [13]:
# ausencia_silv = "(e.VW04090000 + e.VW04100000 + e.VW04110000 = 0)"
ausencia_silv = "(e.VW04090000 IS NULL OR e.VW04090000 = 0) AND (e.VW04100000 IS NULL OR e.VW04100000 = 0) AND (e.VW04110000 IS NULL OR e.VW04110000 = 0)"
join_silv = " AND ".join([f"e.{col} = s.{col}" for col in key_cols])

# Verificando quantos estabelecimentos têm registros na tabela silvicultura, mas não possuem informações de area
query = f"""
    SELECT COUNT(*)
    FROM (
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN silv s
            ON {join_silv}
        WHERE 
            {ausencia_silv}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    ) estabels_com_silv
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 221 -> 0.22%


### Caso 5 - Efetivo da Silvicultura

In [14]:
# ausencia_silv = "(e.VW04090000 + e.VW04100000 + e.VW04110000 = 0)"
ausencia_silv = "(e.VW04090000 IS NULL OR e.VW04090000 = 0) AND (e.VW04100000 IS NULL OR e.VW04100000 = 0) AND (e.VW04110000 IS NULL OR e.VW04110000 = 0)"
join_silv = " AND ".join([f"e.{col} = es.{col}" for col in key_cols])

# Verificando quantos estabelecimentos têm registros na tabela silvicultura, mas não possuem informações de area
query = f"""
    SELECT COUNT(*)
    FROM (
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN efet_silv es
            ON {join_silv}
        WHERE 
            {ausencia_silv}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    ) estabels_com_efet_silv
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 299 -> 0.30%


### Caso 6 - Extração Vegetal

In [15]:
# ausencia_silv = "(e.VW04090000 + e.VW04100000 + e.VW04110000 = 0)"
ausencia_silv = "(e.VW04090000 IS NULL OR e.VW04090000 = 0) AND (e.VW04100000 IS NULL OR e.VW04100000 = 0) AND (e.VW04110000 IS NULL OR e.VW04110000 = 0)"
join_extr_veg = " AND ".join([f"e.{col} = ev.{col}" for col in key_cols])

# Verificando quantos estabelecimentos têm registros na tabela silvicultura, mas não possuem informações de area
query = f"""
    SELECT COUNT(*)
    FROM (
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN extr_veg ev
            ON {join_extr_veg}
        WHERE 
            {ausencia_silv}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    ) estabels_com_efet_extr_veg
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 2496 -> 2.50%


### Caso 7 - Geral

In [16]:
ausencia_lav_per = "(e.VW04020000 IS NULL OR e.VW04020000 = 0)"
ausencia_lav_temp = "(e.VW04030000 IS NULL OR e.VW04030000 = 0)"
ausencia_hort = "(e.VW04030000 IS NULL OR e.VW04030000 = 0)"
ausencia_silv = "(e.VW04090000 IS NULL OR e.VW04090000 = 0) AND (e.VW04100000 IS NULL OR e.VW04100000 = 0) AND (e.VW04110000 IS NULL OR e.VW04110000 = 0)"

join_lav_per = " AND ".join([f"e.{col} = lp.{col}" for col in key_cols])
join_lav_temp = " AND ".join([f"e.{col} = lt.{col}" for col in key_cols])
join_hort = " AND ".join([f"e.{col} = ht.{col}" for col in key_cols])
join_silv = " AND ".join([f"e.{col} = s.{col}" for col in key_cols])
join_efet_silv = " AND ".join([f"e.{col} = es.{col}" for col in key_cols])
join_extr_veg = " AND ".join([f"e.{col} = ev.{col}" for col in key_cols])

query_r3 = f"""
    SELECT 
        COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800)
    FROM (
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN lav_per lp
            ON {join_lav_per}
        WHERE {ausencia_lav_per}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800

        UNION

        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN lav_temp lt
            ON {join_lav_temp}
        WHERE {ausencia_lav_temp}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800

        UNION

        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN hort ht
            ON {join_hort}
        WHERE {ausencia_hort}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800

        UNION

        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN silv s
            ON {join_silv}
        WHERE {ausencia_silv}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800

        UNION

        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN efet_silv es
            ON {join_efet_silv}
        WHERE {ausencia_silv}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800

        UNION

        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
        FROM estabel e
        INNER JOIN extr_veg ev
            ON {join_extr_veg}
        WHERE 
            {ausencia_silv}
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    ) estabels_regra3_geral
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query_r3)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 8753 -> 8.75%


In [17]:
query_r3 = """
SELECT
    DISTINCT V010100, NUM_QUADRA, NUM_FACE, V010800
FROM (
    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM estabel e
    INNER JOIN lav_per lp
        ON e.V010100 = lp.V010100 AND e.NUM_QUADRA = lp.NUM_QUADRA AND e.NUM_FACE = lp.NUM_FACE AND e.V010800 = lp.V010800
    WHERE (e.VW04020000 IS NULL OR e.VW04020000 = 0)

    UNION

    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM estabel e
    INNER JOIN lav_temp lt
        ON e.V010100 = lt.V010100 AND e.NUM_QUADRA = lt.NUM_QUADRA AND e.NUM_FACE = lt.NUM_FACE AND e.V010800 = lt.V010800
    WHERE (e.VW04030000 IS NULL OR e.VW04030000 = 0)

    UNION

    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM estabel e
    INNER JOIN hort ht
        ON e.V010100 = ht.V010100 AND e.NUM_QUADRA = ht.NUM_QUADRA AND e.NUM_FACE = ht.NUM_FACE AND e.V010800 = ht.V010800
    WHERE (e.VW04030000 IS NULL OR e.VW04030000 = 0)

    UNION

    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM estabel e
    INNER JOIN silv s
        ON e.V010100 = s.V010100 AND e.NUM_QUADRA = s.NUM_QUADRA AND e.NUM_FACE = s.NUM_FACE AND e.V010800 = s.V010800
    WHERE (e.VW04090000 IS NULL OR e.VW04090000 = 0)
      AND (e.VW04100000 IS NULL OR e.VW04100000 = 0)
      AND (e.VW04110000 IS NULL OR e.VW04110000 = 0)

    UNION

    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM estabel e
    INNER JOIN efet_silv es
        ON e.V010100 = es.V010100 AND e.NUM_QUADRA = es.NUM_QUADRA AND e.NUM_FACE = es.NUM_FACE AND e.V010800 = es.V010800
    WHERE (e.VW04090000 IS NULL OR e.VW04090000 = 0)
      AND (e.VW04100000 IS NULL OR e.VW04100000 = 0)
      AND (e.VW04110000 IS NULL OR e.VW04110000 = 0)

    UNION

    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM estabel e
    INNER JOIN extr_veg ev
        ON e.V010100 = ev.V010100 AND e.NUM_QUADRA = ev.NUM_QUADRA AND e.NUM_FACE = ev.NUM_FACE AND e.V010800 = ev.V010800
    WHERE (e.VW04090000 IS NULL OR e.VW04090000 = 0)
      AND (e.VW04100000 IS NULL OR e.VW04100000 = 0)
      AND (e.VW04110000 IS NULL OR e.VW04110000 = 0)
) t
"""

## Regra 4

In [18]:
query_r4 = """
SELECT 
    COUNT(*)
FROM estabel
WHERE 
    V04020001 = '2'
    AND (V04020000 IS NULL OR V04020000 = 0)
    AND (V04030000 IS NULL OR V04030000 = 0)
    AND (V04040000 IS NULL OR V04040000 = 0)
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query_r4)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 4 -> 0.00%


In [19]:
query_r4 = """
SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800
FROM estabel
WHERE
    V04020001 = '2'
    AND (V04020000 IS NULL OR V04020000 = 0)
    AND (V04030000 IS NULL OR V04030000 = 0)
    AND (V04040000 IS NULL OR V04040000 = 0)
"""

## Regra 5

In [20]:
query_r5 = """
SELECT 
    COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800)
FROM (
    SELECT 
        V010100, NUM_QUADRA, NUM_FACE, V010800 
    FROM lav_temp 
    WHERE 
        (V34020300 = 0 OR V34020300 IS NULL) AND (V34020500 = 0 OR V34020500 IS NULL)
    
    UNION
    
    SELECT 
        V010100, NUM_QUADRA, NUM_FACE, V010800 
    FROM lav_per 
    WHERE 
        (V35021400 = 0 OR V35021400 IS NULL) AND (V35020900 = 0 OR V35020900 IS NULL)

    UNION

    SELECT 
        V010100, NUM_QUADRA, NUM_FACE, V010800 
    FROM hort 
    WHERE 
        (V37020300 = 0 OR V37020300 IS NULL)

    UNION

    SELECT
        V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM silv
    WHERE
        (V40020300 = 0 OR V40020300 IS NULL)

    UNION

    SELECT
        V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM efet_silv
    WHERE
        (V39020300 = 0 OR V39020300 IS NULL) AND (V39020400 = 0 OR V39020400 IS NULL)

    UNION

    SELECT 
        V010100, NUM_QUADRA, NUM_FACE, V010800 
    FROM extr_veg 
    WHERE 
        (V36020300 = 0 OR V36020300 IS NULL) 
)
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query_r5)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 16900 -> 16.90%


In [21]:
query_r5 = """
SELECT DISTINCT V010100, NUM_QUADRA, NUM_FACE, V010800
FROM (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM lav_temp
    WHERE (V34020300 = 0 OR V34020300 IS NULL) AND (V34020500 = 0 OR V34020500 IS NULL)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM lav_per
    WHERE (V35021400 = 0 OR V35021400 IS NULL) AND (V35020900 = 0 OR V35020900 IS NULL)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM hort
    WHERE (V37020300 = 0 OR V37020300 IS NULL)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM silv
    WHERE (V40020300 = 0 OR V40020300 IS NULL)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM efet_silv
    WHERE (V39020300 = 0 OR V39020300 IS NULL) AND (V39020400 = 0 OR V39020400 IS NULL)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM extr_veg
    WHERE (V36020300 = 0 OR V36020300 IS NULL)
) t
"""

## Regra 6

In [22]:
def marcar_r6(query_r6: dict):
    with sqlite3.connect(DB_CRIT) as conn:
        df_lp = pd.read_sql_query(query_r6["lav_per"], conn)

    if not df_lp.empty:
        df_lp["diff_area_total_area_plantada"] = (
            df_lp["VW04020000"].round(2) - df_lp["total_area_plantada"].round(2)
        )
        df_lp = df_lp.loc[df_lp["diff_area_total_area_plantada"] < 0, :].copy()
        df_lp["diff_area_total_area_plantada"] = df_lp["diff_area_total_area_plantada"].abs()
        p99_lp = df_lp["diff_area_total_area_plantada"].quantile(0.99) if not df_lp.empty else None
        df_r6_1 = df_lp.loc[df_lp["diff_area_total_area_plantada"] > p99_lp, key_cols] if p99_lp is not None else pd.DataFrame(columns=key_cols)
    else:
        df_r6_1 = pd.DataFrame(columns=key_cols)

    with sqlite3.connect(DB_CRIT) as conn:
        df_lt = pd.read_sql_query(query_r6["lav_temp"], conn)

    if not df_lt.empty:
        df_lt["diff_area_total_area_colhida"] = (
            df_lt["VW04020000"].round(2) - df_lt["total_colhida"].round(2)
        )
        df_lt = df_lt.loc[df_lt["diff_area_total_area_colhida"] < 0, :].copy()
        df_lt["diff_area_total_area_colhida"] = df_lt["diff_area_total_area_colhida"].abs()
        p99_lt = df_lt["diff_area_total_area_colhida"].quantile(0.99) if not df_lt.empty else None
        df_r6_2 = df_lt.loc[df_lt["diff_area_total_area_colhida"] > p99_lt, key_cols] if p99_lt is not None else pd.DataFrame(columns=key_cols)
    else:
        df_r6_2 = pd.DataFrame(columns=key_cols)

    aplicar_regra_df("R06", pd.concat([df_r6_1, df_r6_2], ignore_index=True))

In [23]:
query_r6 = {
    "lav_per": """
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800, e.VW04020000,
               SUM(lp.VW35020701) AS total_area_plantada
        FROM estabel e
        INNER JOIN lav_per lp
            ON e.V010100 = lp.V010100 AND e.NUM_QUADRA = lp.NUM_QUADRA AND e.NUM_FACE = lp.NUM_FACE AND e.V010800 = lp.V010800
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    """,
    "lav_temp": """
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800, e.VW04020000,
               SUM(lt.VW34020300) AS total_colhida
        FROM estabel e
        INNER JOIN lav_temp lt
            ON e.V010100 = lt.V010100 AND e.NUM_QUADRA = lt.NUM_QUADRA AND e.NUM_FACE = lt.NUM_FACE AND e.V010800 = lt.V010800
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    """
}

### Caso 1 - Lavoura Permanente

In [24]:
query = """
SELECT 
    e.id, e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800, e.VW04020000, 
    SUM(lp.VW35021400) as total_colhida, 
    SUM(lp.VW35020701) as total_area_plantada
FROM estabel e
INNER JOIN lav_per lp
    ON e.V010100 = lp.V010100 AND e.NUM_QUADRA = lp.NUM_QUADRA AND e.NUM_FACE = lp.NUM_FACE AND e.V010800 = lp.V010800
GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df = pd.read_sql_query(query, conn)

# df['diff_area_total_area_plantada'] = df['VW04020000'].round(2) - df['total_area_plantada'].round(2)
# df = df.loc[df['diff_area_total_area_plantada'] < 0, :].reset_index(drop=True)
# df['diff_area_total_area_plantada'] = df['diff_area_total_area_plantada'].abs()

# #tabela com quartis da diferença entre a área total e a área plantada
# quartis = df['diff_area_total_area_plantada'].quantile([i/100 for i in range(10, 100, 10)] + [0.95, 0.99])
# print(quartis)
# print()
# df_R6_1 = df.loc[df['diff_area_total_area_plantada'] > quartis[0.99], ['id'] + key_cols].reset_index(drop=True)
# shape = df_R6_1.shape[0]

# print(f"Total de registros com diferença entre área total e área plantada maior que o 99º percentil: {shape} -> {shape/QTD_ESTABELECIMENTOS:.2%}")

df['diff_area_total_area_colhida'] = df['VW04020000'].round(2) - df['total_colhida'].round(2)
df = df.loc[df['diff_area_total_area_colhida'] < 0, :].reset_index(drop=True)
df['diff_area_total_area_colhida'] = df['diff_area_total_area_colhida'].abs()

df['razao_diff_area_total_area_colhida'] = df['diff_area_total_area_colhida'] / df['VW04020000']

quartis_razao = df['razao_diff_area_total_area_colhida'].quantile([i/100 for i in range(10, 100, 10)] + [0.95, 0.99])
print(quartis_razao)
print()

df_R6_1 = df.loc[df['razao_diff_area_total_area_colhida'] >= 3, key_cols].reset_index(drop=True)
shape = df_R6_1.shape[0]

print(f"Total de registros com razão entre diferença de área total e área colhida maior ou igual a 3x a área total: {shape} -> {shape/QTD_ESTABELECIMENTOS:.2%}")

0.10     0.036825
0.20     0.100000
0.30     0.209667
0.40     0.340739
0.50     0.600000
0.60     1.000000
0.70     1.000000
0.80     1.000000
0.90     2.000000
0.95     7.042857
0.99    66.960000
Name: razao_diff_area_total_area_colhida, dtype: float64

Total de registros com razão entre diferença de área total e área colhida maior ou igual a 3x a área total: 47 -> 0.05%


### Caso 2 - Lavoura Temporaria

In [25]:
query = """
SELECT 
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800, e.VW04020000, 
    SUM(lt.VW34020300) as total_colhida
FROM estabel e
INNER JOIN lav_temp lt
    ON e.V010100 = lt.V010100 AND e.NUM_QUADRA = lt.NUM_QUADRA AND e.NUM_FACE = lt.NUM_FACE AND e.V010800 = lt.V010800
GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
"""

with sqlite3.connect(DB_CRIT) as conn:
    df = pd.read_sql_query(query, conn)

df['diff_area_total_area_colhida'] = df['VW04020000'].round(2) - df['total_colhida'].round(2)
df = df.loc[df['diff_area_total_area_colhida'] < 0, :].reset_index(drop=True)
df['diff_area_total_area_colhida'] = df['diff_area_total_area_colhida'].abs()

df['razao_diff_area_total_area_colhida'] = df['diff_area_total_area_colhida'] / df['VW04020000']

quartis_razao = df['razao_diff_area_total_area_colhida'].quantile([i/100 for i in range(10, 100, 10)] + [0.95, 0.99])
print(quartis_razao)
print()

df_R6_2 = df.loc[df['razao_diff_area_total_area_colhida'] >= 3, key_cols].reset_index(drop=True)
shape = df_R6_2.shape[0]

print(f"Total de registros com razão entre diferença de área total e área colhida maior ou igual a 3x a área total: {shape} -> {shape/QTD_ESTABELECIMENTOS:.2%}")

0.10        0.500000
0.20        1.000000
0.30        2.000000
0.40        3.000000
0.50        5.551804
0.60       10.500000
0.70       24.015627
0.80       78.999997
0.90      339.080000
0.95     1074.000014
0.99    41412.374564
Name: razao_diff_area_total_area_colhida, dtype: float64

Total de registros com razão entre diferença de área total e área colhida maior ou igual a 3x a área total: 4911 -> 4.91%


## Regra 7

In [26]:
query = """
SELECT
    COUNT(*)
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica IN (332, 338, 298, 326)
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 2416 -> 2.42%


In [27]:
query_r7 = """
SELECT 
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
FROM estabel e
JOIN estabel_criticas ec
  ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE ec.id_critica IN (332, 338, 298, 326)
"""

## Regra 8

Diretriz é condicionada a análise do contexto das observações e dos dados apresentados nos formulários, sendo impossível a realização da análise mediante a exploração direta dos dados.

In [28]:
query_r8 = None

## Regra 9

In [29]:
query_r9 = None

## Regra 10

### Caso 1 - Préco médio acima do esperado

Esse caso é baseado diretamente em um crítica

In [30]:
query = """
SELECT
    id, mensagem
FROM criticas
WHERE
    (mensagem LIKE '%valor%' OR mensagem LIKE '%Preço%')
    AND (mensagem LIKE '%vendido%' OR mensagem LIKE '%vendida%' OR mensagem LIKE '%venda%')
    AND mensagem LIKE '%acima%'
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df_criticas = pd.read_sql_query(query, conn)

In [31]:
query = f"""
SELECT
    COUNT(*) as total
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica IN ({','.join(map(str, df_criticas['id'].to_list()))})
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 7018 -> 7.02%


In [32]:
query_r10_1 = query = f"""
SELECT
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica IN ({','.join(map(str, df_criticas['id'].to_list()))})
"""

### Caso 2 - Produto vendido sem produção registrada

In [33]:
query = """
SELECT 
    (SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) 
     FROM lav_temp 
     WHERE V34020600 IS NOT NULL AND (V34020500 IS NULL OR V34020500 = 0)) as lav_temp_count,
    (SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) 
     FROM lav_per 
     WHERE V35021000 IS NOT NULL AND (V35020900 IS NULL OR V35020900 = 0)) as lav_per_count,
    (SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) 
     FROM hort 
     WHERE V37020400 IS NOT NULL AND (V37020300 IS NULL OR V37020300 = 0)) as hort_count,
    (SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) 
     FROM extr_veg 
     WHERE V36020400 IS NOT NULL AND (V36020300 IS NULL OR V36020300 = 0)) as extr_veg_count,
     (SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) 
      FROM silv 
      WHERE V40020400 IS NOT NULL AND (V40020300 IS NULL OR V40020300 = 0)) as silv_count;
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    result = cursor.fetchone()
    print(f"Total de registros com lav_temp: {result[0]} -> {result[0]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Total de registros com lav_per: {result[1]} -> {result[1]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Total de registros com hort: {result[2]} -> {result[2]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Total de registros com extr_veg: {result[3]} -> {result[3]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Total de registros com silv: {result[4]} -> {result[4]/QTD_ESTABELECIMENTOS:.2%}")

Total de registros com lav_temp: 816 -> 0.82%
Total de registros com lav_per: 1153 -> 1.15%
Total de registros com hort: 28 -> 0.03%
Total de registros com extr_veg: 19 -> 0.02%
Total de registros com silv: 6 -> 0.01%


In [34]:
query = """
SELECT
    COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total_estab
FROM (
    SELECT 
        V010100, NUM_QUADRA, NUM_FACE, V010800 
     FROM lav_temp 
     WHERE 
        V34020600 IS NOT NULL 
        AND (V34020500 IS NULL OR V34020500 = 0)
    
    UNION
        
    SELECT 
        V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM lav_per
    WHERE
        V35021000 IS NOT NULL
        AND (V35020900 IS NULL OR V35020900 = 0)
    
    UNION

    SELECT
        V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM hort
    WHERE 
        V37020400 IS NOT NULL AND 
        (V37020300 IS NULL OR V37020300 = 0)

    UNION

    SELECT
        V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM extr_veg
    WHERE 
        V36020400 IS NOT NULL AND 
        (V36020300 IS NULL OR V36020300 = 0)

    UNION

    SELECT
        V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM silv
    WHERE 
        V40020400 IS NOT NULL AND 
        (V40020300 IS NULL OR V40020300 = 0)
) as estabels_venda_sem_prod
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    result = cursor.fetchone()
    print(f"Total de registros: {result[0]} -> {result[0]/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 1998 -> 2.00%


In [35]:
query_r10_2 = """
SELECT 
    DISTINCT V010100, NUM_QUADRA, NUM_FACE, V010800
FROM (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM lav_temp
    WHERE V34020600 IS NOT NULL AND (V34020500 IS NULL OR V34020500 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM lav_per
    WHERE V35021000 IS NOT NULL AND (V35020900 IS NULL OR V35020900 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM hort
    WHERE V37020400 IS NOT NULL AND (V37020300 IS NULL OR V37020300 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM extr_veg
    WHERE V36020400 IS NOT NULL AND (V36020300 IS NULL OR V36020300 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM silv
    WHERE V40020400 IS NOT NULL AND (V40020300 IS NULL OR V40020300 = 0)
) t
"""

In [36]:
query_r10 = {
    "venda_sem_prod": query_r10_1,
    "venda_sem_prod_detalhado": query_r10_2
}

def marcar_r10(query_r10: dict):
    # with sqlite3.connect(DB_CRIT) as conn:
    #     # Ignorada por superar o limiar de contaminação
    #     df_venda_sem_prod = pd.read_sql_query(query_r10["venda_sem_prod"], conn)

    with sqlite3.connect(DB_CRIT) as conn:
        df_venda_sem_prod_detalhado = pd.read_sql_query(query_r10["venda_sem_prod_detalhado"], conn)

    # df_r10 = df_venda_sem_prod.merge(df_venda_sem_prod_detalhado, on=key_cols, how='outer')
    df_r10 = df_venda_sem_prod_detalhado

    aplicar_regra_df("R10", df_r10)    

## Regra 11

### Lavoura Permanente

In [37]:
query = """
SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total 
FROM lav_per 
WHERE V35010100 IS NOT NULL 
  AND (V35020300 IS NULL OR V35020300 = 0)
  AND (V35020700 IS NULL OR V35020700 = 0)
  AND (V35020900 IS NULL OR V35020900 = 0)
  AND (V35021000 IS NULL OR V35021000 = 0)
  AND (V35021200 IS NULL OR V35021200 = 0)
  AND (V35021300 IS NULL OR V35021300 = 0)
  AND (V35021400 IS NULL OR V35021400 = 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 7144 -> 7.14%


### Lavoura Temporaria

In [38]:
query = """SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total 
FROM lav_temp 
WHERE V34010100 IS NOT NULL
    AND (V34020300 IS NULL OR V34020300 = 0)
    AND (V34020500 IS NULL OR V34020500 = 0)
    AND (V34020600 IS NULL OR V34020600 = 0)
    AND (V34020900 IS NULL OR V34020900 = 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 671 -> 0.67%


### Extração Vegetal

In [39]:
query = """SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total 
FROM extr_veg 
WHERE V36010100 IS NOT NULL
    AND (V36020300 IS NULL OR V36020300 = 0)
    AND (V36020400 IS NULL OR V36020400 = 0)
    AND (V36020600 IS NULL OR V36020600 = 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 25 -> 0.03%


### Horticultura

In [40]:
query = """SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total 
FROM hort
WHERE V37020100 IS NOT NULL
    AND (V37020300 IS NULL OR V37020300 = 0)
    AND (V37020400 IS NULL OR V37020400 = 0)
    AND (V37020600 IS NULL OR V37020600 = 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 27 -> 0.03%


### Efetivo da Silvicultura

In [41]:
query = """SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total 
FROM efet_silv
WHERE V39010100 IS NOT NULL
    AND (V39020300 IS NULL OR V39020300 = 0)
    AND (V39020400 IS NULL OR V39020400 = 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 0 -> 0.00%


### Silvicultura

In [42]:
query = """SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total 
FROM silv 
WHERE V40010100 IS NOT NULL
    AND (V40020300 IS NULL OR V40020300 = 0)
    AND (V40020400 IS NULL OR V40020400 = 0)
    AND (V40020600 IS NULL OR V40020600 = 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 6 -> 0.01%


### Agroindustria

In [43]:
query = """SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total 
FROM agro_ind
WHERE V41010200 IS NOT NULL
    AND (V41030300 IS NULL OR V41030300 = 0)
    AND (V41030500 IS NULL OR V41030500 = 0)
    AND (V41030700 IS NULL OR V41030700 = 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 0 -> 0.00%


### Caso Geral

In [44]:
query = """
SELECT 
    COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total
FROM (
SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800  
FROM lav_per 
WHERE V35010100 IS NOT NULL 
  AND (V35020300 IS NULL OR V35020300 = 0)
  AND (V35020700 IS NULL OR V35020700 = 0)
  AND (V35020900 IS NULL OR V35020900 = 0)
  AND (V35021000 IS NULL OR V35021000 = 0)
  AND (V35021200 IS NULL OR V35021200 = 0)
  AND (V35021300 IS NULL OR V35021300 = 0)
  AND (V35021400 IS NULL OR V35021400 = 0)

UNION

SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800 
FROM lav_temp 
WHERE V34010100 IS NOT NULL
    AND (V34020300 IS NULL OR V34020300 = 0)
    AND (V34020500 IS NULL OR V34020500 = 0)
    AND (V34020600 IS NULL OR V34020600 = 0)
    AND (V34020900 IS NULL OR V34020900 = 0)

UNION

SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800 
FROM extr_veg 
WHERE V36010100 IS NOT NULL
    AND (V36020300 IS NULL OR V36020300 = 0)
    AND (V36020400 IS NULL OR V36020400 = 0)
    AND (V36020600 IS NULL OR V36020600 = 0)

UNION

SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800
FROM hort
WHERE V37020100 IS NOT NULL
    AND (V37020300 IS NULL OR V37020300 = 0)
    AND (V37020400 IS NULL OR V37020400 = 0)
    AND (V37020600 IS NULL OR V37020600 = 0)

UNION

SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800
FROM efet_silv
WHERE V39010100 IS NOT NULL
    AND (V39020300 IS NULL OR V39020300 = 0)
    AND (V39020400 IS NULL OR V39020400 = 0)

UNION

SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800
FROM silv 
WHERE V40010100 IS NOT NULL
    AND (V40020300 IS NULL OR V40020300 = 0)
    AND (V40020400 IS NULL OR V40020400 = 0)
    AND (V40020600 IS NULL OR V40020600 = 0)

UNION

SELECT
    V010100, NUM_QUADRA, NUM_FACE, V010800
FROM agro_ind
WHERE V41010200 IS NOT NULL
    AND (V41030300 IS NULL OR V41030300 = 0)
    AND (V41030500 IS NULL OR V41030500 = 0)
    AND (V41030700 IS NULL OR V41030700 = 0)
)"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 7834 -> 7.83%


In [45]:
query_r11 = """
SELECT 
    DISTINCT V010100, NUM_QUADRA, NUM_FACE, V010800
FROM (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM lav_per
    WHERE V35010100 IS NOT NULL
      AND (V35020300 IS NULL OR V35020300 = 0)
      AND (V35020700 IS NULL OR V35020700 = 0)
      AND (V35020900 IS NULL OR V35020900 = 0)
      AND (V35021000 IS NULL OR V35021000 = 0)
      AND (V35021200 IS NULL OR V35021200 = 0)
      AND (V35021300 IS NULL OR V35021300 = 0)
      AND (V35021400 IS NULL OR V35021400 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM lav_temp
    WHERE V34010100 IS NOT NULL
      AND (V34020300 IS NULL OR V34020300 = 0)
      AND (V34020500 IS NULL OR V34020500 = 0)
      AND (V34020600 IS NULL OR V34020600 = 0)
      AND (V34020900 IS NULL OR V34020900 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM extr_veg
    WHERE V36010100 IS NOT NULL
      AND (V36020300 IS NULL OR V36020300 = 0)
      AND (V36020400 IS NULL OR V36020400 = 0)
      AND (V36020600 IS NULL OR V36020600 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM hort
    WHERE V37020100 IS NOT NULL
      AND (V37020300 IS NULL OR V37020300 = 0)
      AND (V37020400 IS NULL OR V37020400 = 0)
      AND (V37020600 IS NULL OR V37020600 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM efet_silv
    WHERE V39010100 IS NOT NULL
      AND (V39020300 IS NULL OR V39020300 = 0)
      AND (V39020400 IS NULL OR V39020400 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM silv
    WHERE V40010100 IS NOT NULL
      AND (V40020300 IS NULL OR V40020300 = 0)
      AND (V40020400 IS NULL OR V40020400 = 0)
      AND (V40020600 IS NULL OR V40020600 = 0)

    UNION

    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM agro_ind
    WHERE V41010200 IS NOT NULL
      AND (V41030300 IS NULL OR V41030300 = 0)
      AND (V41030500 IS NULL OR V41030500 = 0)
      AND (V41030700 IS NULL OR V41030700 = 0)
) t
"""

## Regra 12

### Caso 1 - Vacas reprodutoras sem produção de leite

In [46]:
query = """
SELECT 
    V14090302 as qtd_vacas_reprodutoras, 
    V14020101 as finalidade_criacao
FROM pecu WHERE 
    V14090302 > 0 AND V14160000 = '1';
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df = pd.read_sql_query(query, conn)

quartis = df['qtd_vacas_reprodutoras'].quantile([i/100 for i in range(10, 100, 10)] + [0.95, 0.99])
print(quartis)
print()
shape = df[df['qtd_vacas_reprodutoras'] > quartis[0.99]].shape[0]

print(f"Total de registros com mais de {quartis[0.99]} vacas reprodutoras: {shape} -> {shape/QTD_ESTABELECIMENTOS:.2%}")

0.10      3.0
0.20      5.0
0.30      8.0
0.40     12.0
0.50     20.0
0.60     29.0
0.70     40.0
0.80     70.0
0.90    150.0
0.95    292.0
0.99    872.0
Name: qtd_vacas_reprodutoras, dtype: float64

Total de registros com mais de 872.0 vacas reprodutoras: 311 -> 0.31%


### Caso 2 - Bovinos declarados com a finalidade de trabalho

In [47]:
query = """
SELECT 
    p.V14010101 as qtd_bovinos, 
    e.VW01170300 as area_total_estabel,
    e.VW04020000 as area_lav_perm,
    e.VW04030000 as area_lav_temp,
    e.VW04160000 as area_lavouras
FROM estabel e
JOIN pecu p
    ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA AND 
    e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
WHERE 
    p.V14020101 = '6';
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df = pd.read_sql_query(query, conn)

df['relacao_area_qtd_bovinos'] = df['area_total_estabel'] / df['qtd_bovinos']
df['relacao_area_lav_qtd_bovinos'] = df['area_lavouras'] / df['qtd_bovinos']

quartis = df['relacao_area_qtd_bovinos'].quantile([i/100 for i in range(10, 100, 10)])
print(quartis)
print()
shape = df[df['relacao_area_qtd_bovinos'] < quartis[0.10]].shape[0]

print(f"Total de registros com relação entre área total e quantidade de bovinos menor que o 10º percentil: {shape} -> {shape/QTD_ESTABELECIMENTOS:.2%}")


quartis = df['relacao_area_lav_qtd_bovinos'].quantile([i/100 for i in range(10, 100, 10)])
print(quartis)
print()
shape = df[df['relacao_area_lav_qtd_bovinos'] < quartis[0.60]].shape[0]

print(f"Total de registros com relação entre área lavrada e quantidade de bovinos menor que o 60º percentil: {shape} -> {shape/QTD_ESTABELECIMENTOS:.2%}")

0.1    0.511406
0.2    0.663771
0.3    0.826667
0.4    1.000000
0.5    1.245619
0.6    1.489215
0.7    1.936000
0.8    2.742667
0.9    4.576442
Name: relacao_area_qtd_bovinos, dtype: float64

Total de registros com relação entre área total e quantidade de bovinos menor que o 10º percentil: 70 -> 0.07%
0.1    0.000000
0.2    0.000000
0.3    0.000000
0.4    0.000000
0.5    0.000000
0.6    0.001812
0.7    0.021833
0.8    0.050000
0.9    0.142857
Name: relacao_area_lav_qtd_bovinos, dtype: float64

Total de registros com relação entre área lavrada e quantidade de bovinos menor que o 60º percentil: 417 -> 0.42%


In [48]:
query_r12 = {
    "case1": """
        SELECT V010100, NUM_QUADRA, NUM_FACE, V010800,
               V14090302 AS qtd_vacas_reprodutoras
        FROM pecu
        WHERE V14090302 > 0 AND V14160000 = '1'
    """,
    "case2": """
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800,
               p.V14010101 AS qtd_bovinos,
               e.VW01170300 AS area_total_estabel,
               e.VW04160000 AS area_lavouras
        FROM estabel e
        JOIN pecu p
          ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA
         AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
        WHERE p.V14020101 = '6' AND p.V14010101 > 0
    """
}

def marcar_r12(query_r12: dict):
    with sqlite3.connect(DB_MAIN) as conn:
        df_case1 = pd.read_sql_query(query_r12["case1"], conn)

    if not df_case1.empty:
        p99_vacas = df_case1["qtd_vacas_reprodutoras"].quantile(0.99)
        df_case1 = df_case1.loc[df_case1["qtd_vacas_reprodutoras"] > p99_vacas, key_cols]
    else:
        df_case1 = pd.DataFrame(columns=key_cols)

    with sqlite3.connect(DB_MAIN) as conn:
        df_case2 = pd.read_sql_query(query_r12["case2"], conn)

    if not df_case2.empty:
        df_case2["relacao_area_qtd_bovinos"] = df_case2["area_total_estabel"] / df_case2["qtd_bovinos"]
        df_case2["relacao_area_lav_qtd_bovinos"] = df_case2["area_lavouras"] / df_case2["qtd_bovinos"]

        p10_rel_total = df_case2["relacao_area_qtd_bovinos"].quantile(0.10)
        p60_rel_lav = df_case2["relacao_area_lav_qtd_bovinos"].quantile(0.60)

        df_case2 = df_case2.loc[
            (df_case2["relacao_area_qtd_bovinos"] < p10_rel_total)
            | (df_case2["relacao_area_lav_qtd_bovinos"] < p60_rel_lav),
            key_cols
        ]
    else:
        df_case2 = pd.DataFrame(columns=key_cols)

    aplicar_regra_df("R12", pd.concat([df_case1, df_case2], ignore_index=True))

## Regra 13

In [49]:
query = """
SELECT 
    COUNT(*) as total
FROM pecu p
JOIN estabel e ON p.V010100 = e.V010100 AND p.NUM_QUADRA = e.NUM_QUADRA AND p.NUM_FACE = e.NUM_FACE AND p.V010800 = e.V010800
WHERE 
(p.V22020401 > 0)
AND (
    e.V13011400 = '2' OR e.V13011500 = '2' OR e.V13011600 = '2' OR e.V13011700 = '2' OR 
    e.V13011800 = '2' OR e.V13011900 = '2' OR e.V13012000 = '2' OR e.V13012100 = '2' OR 
    e.V13012200 = '2' OR e.V13012500 = '2' OR e.V13012400 = '2' OR e.V13012700 = '2' OR 
    e.V13012800 = '2' OR e.V13012900 = '2' OR e.V13013000 = '2' OR e.V13013100 = '2'
);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    result = cursor.fetchone()
    print(f"Total de registros: {result[0]} -> {result[0]/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 556 -> 0.56%


In [50]:
query = """
SELECT 
    COUNT(*) as total
FROM pecu
WHERE 
    V22020401 < 1000
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    result = cursor.fetchone()
    print(f"Total de registros: {result[0]} -> {result[0]/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 508 -> 0.51%


In [51]:
query = """
SELECT 
    COUNT(*) as total
FROM pecu
WHERE 
    V22020401 > 10000
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    result = cursor.fetchone()
    print(f"Total de registros: {result[0]} -> {result[0]/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 54 -> 0.05%


In [52]:
query_r13 = """
SELECT 
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
FROM pecu p
JOIN estabel e
  ON p.V010100 = e.V010100 AND p.NUM_QUADRA = e.NUM_QUADRA
 AND p.NUM_FACE = e.NUM_FACE AND p.V010800 = e.V010800
WHERE p.V22020401 > 0
  AND (
      e.V13011400 = '2' OR e.V13011500 = '2' OR e.V13011600 = '2' OR e.V13011700 = '2' OR
      e.V13011800 = '2' OR e.V13011900 = '2' OR e.V13012000 = '2' OR e.V13012100 = '2' OR
      e.V13012200 = '2' OR e.V13012400 = '2' OR e.V13012500 = '2' OR e.V13012700 = '2' OR
      e.V13012800 = '2' OR e.V13012900 = '2' OR e.V13013000 = '2' OR e.V13013100 = '2'
  )
"""

## Regra 14

In [53]:
query = """
SELECT
    COUNT(*) as total
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica = 226
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 5092 -> 5.09%


In [54]:
query_r14 = """
SELECT 
e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
FROM estabel e
JOIN estabel_criticas ec
  ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE ec.id_critica = 226
"""

## Regra 15

### Caso 1 - Variáveis de existência

In [55]:
query = """
SELECT 
    SUM(CASE WHEN (V19010000 = '2' AND (V19010101 IS NULL OR V19010101 = 0)) THEN 1 ELSE 0 END) as suinos_inconsistentes,
    SUM(CASE WHEN (V22010000 = '2' AND (V22010101 IS NULL OR V22010101 = 0)) THEN 1 ELSE 0 END) as aves_inconsistentes,
    SUM(CASE WHEN (V14010000 = '2' AND (V14010101 IS NULL OR V14010101 = 0)) THEN 1 ELSE 0 END) as bovinos_inconsistentes,
    SUM(CASE WHEN (V15010000 = '2' AND (V15010101 IS NULL OR V15010101 = 0)) THEN 1 ELSE 0 END) as bubalinos_inconsistentes,
    SUM(CASE WHEN (V16010000 = '2' AND (V16010101 IS NULL OR V16010101 = 0)) THEN 1 ELSE 0 END) as equinos_inconsistentes,
    SUM(CASE WHEN (V17010000 = '2' AND (V17010101 IS NULL OR V17010101 = 0)) THEN 1 ELSE 0 END) as asininos_inconsistentes,
    SUM(CASE WHEN (V18010000 = '2' AND (V18010101 IS NULL OR V18010101 = 0)) THEN 1 ELSE 0 END) as muares_inconsistentes,
    SUM(CASE WHEN (V20010000 = '2' AND (V20010101 IS NULL OR V20010101 = 0)) THEN 1 ELSE 0 END) as caprinos_inconsistentes,
    SUM(CASE WHEN (V21010000 = '2' AND (V21010101 IS NULL OR V21010101 = 0)) THEN 1 ELSE 0 END) as ovinos_inconsistentes,
    SUM(CASE WHEN (V25010000 = '2' AND (V25010101 IS NULL OR V25010101 = 0)) THEN 1 ELSE 0 END) as codornas_inconsistentes,
    SUM(CASE WHEN (V27010000 = '2' AND (V27010101 IS NULL OR V27010101 = 0)) THEN 1 ELSE 0 END) as coelhos_inconsistentes,
    SUM(CASE WHEN (V26010000 = '2' AND (V26010101 IS NULL OR V26010101 = 0) AND (V24010101 IS NULL OR V24010101 = 0) AND (V23010101 IS NULL OR V23010101 = 0)) THEN 1 ELSE 0 END) as outras_aves_inconsistentes,
    SUM(CASE WHEN (V28080000 = '2' AND (V28080101 IS NULL OR V28080101 = 0)) THEN 1 ELSE 0 END) as abelhas_inconsistentes
FROM pecu
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    result = cursor.fetchone()
    print(f"Suínos inconsistentes: {result[0]} -> {result[0]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Aves inconsistentes: {result[1]} -> {result[1]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Bovinos inconsistentes: {result[2]} -> {result[2]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Bubalinos inconsistentes: {result[3]} -> {result[3]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Equinos inconsistentes: {result[4]} -> {result[4]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Asininos inconsistentes: {result[5]} -> {result[5]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Muares inconsistentes: {result[6]} -> {result[6]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Caprinos inconsistentes: {result[7]} -> {result[7]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Ovinos inconsistentes: {result[8]} -> {result[8]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Codornas inconsistentes: {result[9]} -> {result[9]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Coelhos inconsistentes: {result[10]} -> {result[10]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Outras aves com inconsistentes (com outras inconsistências): {result[11]} -> {result[11]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Abelhas inconsistentes: {result[12]} -> {result[12]/QTD_ESTABELECIMENTOS:.2%}")

Suínos inconsistentes: 0 -> 0.00%
Aves inconsistentes: 0 -> 0.00%
Bovinos inconsistentes: 26 -> 0.03%
Bubalinos inconsistentes: 0 -> 0.00%
Equinos inconsistentes: 3 -> 0.00%
Asininos inconsistentes: 0 -> 0.00%
Muares inconsistentes: 0 -> 0.00%
Caprinos inconsistentes: 0 -> 0.00%
Ovinos inconsistentes: 1 -> 0.00%
Codornas inconsistentes: 0 -> 0.00%
Coelhos inconsistentes: 0 -> 0.00%
Outras aves com inconsistentes (com outras inconsistências): 0 -> 0.00%
Abelhas inconsistentes: 0 -> 0.00%


In [56]:
query = """
SELECT
    COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total_estab
FROM pecu
WHERE
    V19010000 = '2' AND (V19010101 IS NULL OR V19010101 = 0) OR
    V22010000 = '2' AND (V22010101 IS NULL OR V22010101 = 0) OR
    V14010000 = '2' AND (V14010101 IS NULL OR V14010101 = 0) OR
    V15010000 = '2' AND (V15010101 IS NULL OR V15010101 = 0) OR
    V16010000 = '2' AND (V16010101 IS NULL OR V16010101 = 0) OR
    V17010000 = '2' AND (V17010101 IS NULL OR V17010101 = 0) OR
    V18010000 = '2' AND (V18010101 IS NULL OR V18010101 = 0) OR
    V20010000 = '2' AND (V20010101 IS NULL OR V20010101 = 0) OR
    V21010000 = '2' AND (V21010101 IS NULL OR V21010101 = 0) OR
    V25010000 = '2' AND (V25010101 IS NULL OR V25010101 = 0) OR
    V27010000 = '2' AND (V27010101 IS NULL OR V27010101 = 0) OR
    V26010000 = '2' AND ((V26010101 IS NULL OR V26010101 = 0) AND (V24010101 IS NULL OR V24010101 = 0) AND (V23010101 IS NULL OR V23010101 = 0)) OR
    V28080000 = '2' AND (V28080101 IS NULL OR V28080101 = 0)
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 30 -> 0.03%


### Caso 2 - Quadro das caracteristicas da pecuária

In [57]:
# V13012900 -> aquicultura => complexo de identificar a relação de existência de produção, precisa de refino
# V13013000 -> aquicultura => complexo de identificar a relação de existência de produção, precisa de refino
# V13013200 -> pesca ou apanha de crustaceos => complexo de identificar a relação de existência de produção, precisa de refino

query = """
SELECT 
    SUM(CASE WHEN (e.V13011900 = '2' AND (p.V19010101 IS NULL OR p.V19010101 = 0)) THEN 1 ELSE 0 END) as suinos_inconsistentes,
    SUM(CASE WHEN (e.V13012200 = '2' AND (p.V22010101 IS NULL OR p.V22010101 = 0)) THEN 1 ELSE 0 END) as aves_inconsistentes,
    SUM(CASE WHEN (e.V13011400 = '2' AND (p.V14010101 IS NULL OR p.V14010101 = 0)) THEN 1 ELSE 0 END) as bovinos_inconsistentes,
    SUM(CASE WHEN (e.V13011500 = '2' AND (p.V15010101 IS NULL OR p.V15010101 = 0)) THEN 1 ELSE 0 END) as bubalinos_inconsistentes,
    SUM(CASE WHEN (e.V13011600 = '2' AND (p.V16010101 IS NULL OR p.V16010101 = 0)) THEN 1 ELSE 0 END) as equinos_inconsistentes,
    SUM(CASE WHEN (e.V13011700 = '2' AND (p.V17010101 IS NULL OR p.V17010101 = 0)) THEN 1 ELSE 0 END) as asininos_inconsistentes,
    SUM(CASE WHEN (e.V13011800 = '2' AND (p.V18010101 IS NULL OR p.V18010101 = 0)) THEN 1 ELSE 0 END) as muares_inconsistentes,
    SUM(CASE WHEN (e.V13012000 = '2' AND (p.V20010101 IS NULL OR p.V20010101 = 0)) THEN 1 ELSE 0 END) as caprinos_inconsistentes,
    SUM(CASE WHEN (e.V13012100 = '2' AND (p.V21010101 IS NULL OR p.V21010101 = 0)) THEN 1 ELSE 0 END) as ovinos_inconsistentes,
    SUM(CASE WHEN (e.V13012500 = '2' AND (p.V25010101 IS NULL OR p.V25010101 = 0)) THEN 1 ELSE 0 END) as codornas_inconsistentes,
    SUM(CASE WHEN (e.V13012700 = '2' AND (p.V27010101 IS NULL OR p.V27010101 = 0)) THEN 1 ELSE 0 END) as coelhos_inconsistentes,
    SUM(CASE WHEN (e.V13012400 = '2' AND (p.V26010101 IS NULL OR p.V26010101 = 0) AND (p.V24010101 IS NULL OR p.V24010101 = 0) AND (p.V23010101 IS NULL OR p.V23010101 = 0)) THEN 1 ELSE 0 END) as outras_aves_inconsistentes,
    SUM(CASE WHEN (e.V13012800 = '2' AND (p.V28080101 IS NULL OR p.V28080101 = 0)) THEN 1 ELSE 0 END) as abelhas_inconsistentes,
    SUM(CASE WHEN (e.V13013100 = '2' AND (p.V31010101 IS NULL OR p.V31010101 = 0)) THEN 1 ELSE 0 END) as bicho_seda_inconsistentes
FROM pecu p
JOIN estabel e ON p.V010100 = e.V010100 AND p.NUM_QUADRA = e.NUM_QUADRA AND p.NUM_FACE = e.NUM_FACE AND p.V010800 = e.V010800
"""

# V26010000

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    result = cursor.fetchone()
    print(f"Suínos inconsistentes: {result[0]} -> {result[0]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Aves inconsistentes: {result[1]} -> {result[1]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Bovinos inconsistentes: {result[2]} -> {result[2]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Bubalinos inconsistentes: {result[3]} -> {result[3]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Equinos inconsistentes: {result[4]} -> {result[4]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Asininos inconsistentes: {result[5]} -> {result[5]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Muares inconsistentes: {result[6]} -> {result[6]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Caprinos inconsistentes: {result[7]} -> {result[7]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Ovinos inconsistentes: {result[8]} -> {result[8]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Codornas inconsistentes: {result[9]} -> {result[9]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Coelhos inconsistentes: {result[10]} -> {result[10]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Outras aves com inconsistentes (com outras inconsistências): {result[11]} -> {result[11]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Abelhas inconsistentes: {result[12]} -> {result[12]/QTD_ESTABELECIMENTOS:.2%}")
    print(f"Bicho-da-seda inconsistentes: {result[13]} -> {result[13]/QTD_ESTABELECIMENTOS:.2%}")

Suínos inconsistentes: 324 -> 0.32%
Aves inconsistentes: 243 -> 0.24%
Bovinos inconsistentes: 520 -> 0.52%
Bubalinos inconsistentes: 5 -> 0.01%
Equinos inconsistentes: 78 -> 0.08%
Asininos inconsistentes: 18 -> 0.02%
Muares inconsistentes: 28 -> 0.03%
Caprinos inconsistentes: 89 -> 0.09%
Ovinos inconsistentes: 149 -> 0.15%
Codornas inconsistentes: 5 -> 0.01%
Coelhos inconsistentes: 2 -> 0.00%
Outras aves com inconsistentes (com outras inconsistências): 96 -> 0.10%
Abelhas inconsistentes: 29 -> 0.03%
Bicho-da-seda inconsistentes: 0 -> 0.00%


In [58]:
query = """
SELECT
    COUNT(DISTINCT e.V010100 || e.NUM_QUADRA || e.NUM_FACE || e.V010800) as total_estab
FROM pecu p
JOIN estabel e ON p.V010100 = e.V010100 AND p.NUM_QUADRA = e.NUM_QUADRA AND p.NUM_FACE = e.NUM_FACE AND p.V010800 = e.V010800
WHERE
    e.V13011900 = '2' AND (p.V19010101 IS NULL OR p.V19010101 = 0) OR
    e.V13012200 = '2' AND (p.V22010101 IS NULL OR p.V22010101 = 0) OR
    e.V13011400 = '2' AND (p.V14010101 IS NULL OR p.V14010101 = 0) OR
    e.V13011500 = '2' AND (p.V15010101 IS NULL OR p.V15010101 = 0) OR
    e.V13011600 = '2' AND (p.V16010101 IS NULL OR p.V16010101 = 0) OR
    e.V13011700 = '2' AND (p.V17010101 IS NULL OR p.V17010101 = 0) OR
    e.V13011800 = '2' AND (p.V18010101 IS NULL OR p.V18010101 = 0) OR
    e.V13012000 = '2' AND (p.V20010101 IS NULL OR p.V20010101 = 0) OR
    e.V13012100 = '2' AND (p.V21010101 IS NULL OR p.V21010101 = 0) OR
    e.V13012500 = '2' AND (p.V25010101 IS NULL OR p.V25010101 = 0) OR
    e.V13012700 = '2' AND (p.V27010101 IS NULL OR p.V27010101 = 0) OR
    e.V13012400 = '2' AND ((p.V26010101 IS NULL OR p.V26010101 = 0) AND (p.V24010101 IS NULL OR p.V24010101 = 0) AND (p.V23010101 IS NULL OR p.V23010101 = 0)) OR
    e.V13012800 = '2' AND (p.V28080101 IS NULL OR p.V28080101 = 0) OR
    e.V13013100 = '2' AND (p.V31010101 IS NULL OR p.V31010101 = 0)
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 1416 -> 1.42%


In [59]:
query_r15 = """
SELECT 
    DISTINCT V010100, NUM_QUADRA, NUM_FACE, V010800
FROM (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM pecu
    WHERE
        (V19010000 = '2' AND (V19010101 IS NULL OR V19010101 = 0)) OR
        (V22010000 = '2' AND (V22010101 IS NULL OR V22010101 = 0)) OR
        (V14010000 = '2' AND (V14010101 IS NULL OR V14010101 = 0)) OR
        (V15010000 = '2' AND (V15010101 IS NULL OR V15010101 = 0)) OR
        (V16010000 = '2' AND (V16010101 IS NULL OR V16010101 = 0)) OR
        (V17010000 = '2' AND (V17010101 IS NULL OR V17010101 = 0)) OR
        (V18010000 = '2' AND (V18010101 IS NULL OR V18010101 = 0)) OR
        (V20010000 = '2' AND (V20010101 IS NULL OR V20010101 = 0)) OR
        (V21010000 = '2' AND (V21010101 IS NULL OR V21010101 = 0)) OR
        (V25010000 = '2' AND (V25010101 IS NULL OR V25010101 = 0)) OR
        (V27010000 = '2' AND (V27010101 IS NULL OR V27010101 = 0)) OR
        (V26010000 = '2' AND ((V26010101 IS NULL OR V26010101 = 0) AND (V24010101 IS NULL OR V24010101 = 0) AND (V23010101 IS NULL OR V23010101 = 0))) OR
        (V28080000 = '2' AND (V28080101 IS NULL OR V28080101 = 0))

    UNION

    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM pecu p
    JOIN estabel e
      ON p.V010100 = e.V010100 AND p.NUM_QUADRA = e.NUM_QUADRA
     AND p.NUM_FACE = e.NUM_FACE AND p.V010800 = e.V010800
    WHERE
        (e.V13011900 = '2' AND (p.V19010101 IS NULL OR p.V19010101 = 0)) OR
        (e.V13012200 = '2' AND (p.V22010101 IS NULL OR p.V22010101 = 0)) OR
        (e.V13011400 = '2' AND (p.V14010101 IS NULL OR p.V14010101 = 0)) OR
        (e.V13011500 = '2' AND (p.V15010101 IS NULL OR p.V15010101 = 0)) OR
        (e.V13011600 = '2' AND (p.V16010101 IS NULL OR p.V16010101 = 0)) OR
        (e.V13011700 = '2' AND (p.V17010101 IS NULL OR p.V17010101 = 0)) OR
        (e.V13011800 = '2' AND (p.V18010101 IS NULL OR p.V18010101 = 0)) OR
        (e.V13012000 = '2' AND (p.V20010101 IS NULL OR p.V20010101 = 0)) OR
        (e.V13012100 = '2' AND (p.V21010101 IS NULL OR p.V21010101 = 0)) OR
        (e.V13012500 = '2' AND (p.V25010101 IS NULL OR p.V25010101 = 0)) OR
        (e.V13012700 = '2' AND (p.V27010101 IS NULL OR p.V27010101 = 0)) OR
        (e.V13012400 = '2' AND ((p.V26010101 IS NULL OR p.V26010101 = 0) AND (p.V24010101 IS NULL OR p.V24010101 = 0) AND (p.V23010101 IS NULL OR p.V23010101 = 0))) OR
        (e.V13012800 = '2' AND (p.V28080101 IS NULL OR p.V28080101 = 0)) OR
        (e.V13013100 = '2' AND (p.V31010101 IS NULL OR p.V31010101 = 0))
) t
"""

## Regra 16

### Caso 1 - Sem declaração de gastos

In [60]:
query = """
SELECT 
    COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total
FROM estabel
WHERE
    V13080102 = '2' 
    AND (V45011500 = 0 OR V45011500 IS NULL);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 813 -> 0.81%


### Caso 2 - Sem declaração de gastos e nem culturas para alimentação de animais

In [61]:
query = """
SELECT 
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800,
    GROUP_CONCAT(l.V34010100) as V34010100
FROM estabel e
LEFT JOIN lav_temp l ON e.V010100 = l.V010100 AND e.NUM_QUADRA = l.NUM_QUADRA AND e.NUM_FACE = l.NUM_FACE AND e.V010800 = l.V010800
WHERE
    e.V13080102 = '2' 
    AND (e.V45011500 = 0 OR e.V45011500 IS NULL)
GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df = pd.read_sql_query(query, conn)


cods = {262, 263, 266, 264}
def parse_codes(v):
    if v is None or str(v).strip() == "":
        return set()
    return {int(x.strip()) for x in str(v).split(",") if x.strip()}

df_sem_intersecao = df[df["V34010100"].apply(lambda v: len(parse_codes(v) & cods) == 0)]
df_sem_intersecao

,V010100,NUM_QUADRA,NUM_FACE,V010800,V34010100
0,110008015000005,0,231,224,237
1,110008015000009,0,3,1,215
2,110015525000007,0,60,60,None
3,110020505000045,0,159,156,"237,240"
4,110020505000093,0,14,14,237
...,...,...,...,...,...
807,522000905000010,0,37,37,"215,240,242"
808,522050405000009,0,35,35,None
809,522145205000005,0,161,161,242
810,522180905000005,0,211,196,None


In [62]:
query_r16 = {
    "case1": """
        SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
        FROM estabel
        WHERE V13080102 = '2' AND (V45011500 = 0 OR V45011500 IS NULL)
    """,
    "case2": """
        SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800,
               GROUP_CONCAT(l.V34010100) AS V34010100
        FROM estabel e
        LEFT JOIN lav_temp l
          ON e.V010100 = l.V010100 AND e.NUM_QUADRA = l.NUM_QUADRA
         AND e.NUM_FACE = l.NUM_FACE AND e.V010800 = l.V010800
        WHERE e.V13080102 = '2' AND (e.V45011500 = 0 OR e.V45011500 IS NULL)
        GROUP BY e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    """
}

def marcar_r16(query_r16: dict):
    df_case1 = fetch_keys(query_r16["case1"], DB_MAIN)

    with sqlite3.connect(DB_MAIN) as conn:
        df_case2 = pd.read_sql_query(query_r16["case2"], conn)

    cods = {262, 263, 264, 266}

    def parse_codes(v):
        if v is None or str(v).strip() == "":
            return set()
        return {int(x.strip()) for x in str(v).split(",") if x.strip()}

    if not df_case2.empty:
        df_case2 = df_case2.loc[
            df_case2["V34010100"].apply(lambda v: len(parse_codes(v) & cods) == 0),
            key_cols
        ]
    else:
        df_case2 = pd.DataFrame(columns=key_cols)

    aplicar_regra_df("R16", pd.concat([df_case1, df_case2], ignore_index=True))

## Regra 17

In [63]:
query = """
SELECT 
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800,
    e.VW01170300 as area_total_estabel,
    e.VW04050000 as area_pastagem_natural, 
    e.VW04060000 as area_pastagem_plantada,
    e.VW04070000 as area_pastagem_degradada,
    e.VW04170000 as area_pastagem_total
FROM estabel e 
LEFT JOIN pecu p ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
WHERE 
    (e.VW04050000 > 0 OR e.VW04060000 > 0)
    AND (p.V14010000 = '1' OR p.V14010000 IS NULL)
    AND (p.V15010000 = '1' OR p.V15010000 IS NULL)
    AND (p.V20010000 = '1' OR p.V20010000 IS NULL)
    AND (p.V21010000 = '1' OR p.V21010000 IS NULL)
    AND (p.V18010000 = '1' OR p.V18010000 IS NULL)
    AND (p.V16010000 = '1' OR p.V16010000 IS NULL)
    AND (p.V17010000 = '1' OR p.V17010000 IS NULL);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df = pd.read_sql_query(query, conn)

df["percent_area_pastagem"] = df["area_pastagem_total"] / df["area_total_estabel"]
print(df["percent_area_pastagem"].describe())

df["percent_area_pastagem_boa"] = (df["area_pastagem_natural"] + df["area_pastagem_plantada"]) / df["area_total_estabel"]
print(df["percent_area_pastagem_boa"].describe())

count    5504.000000
mean        0.375298
std         0.299119
min         0.000200
25%         0.104167
50%         0.300000
75%         0.606061
max         1.000000
Name: percent_area_pastagem, dtype: float64
count    562.000000
mean       0.478480
std        0.272546
min        0.005600
25%        0.250087
50%        0.486018
75%        0.700000
max        1.000000
Name: percent_area_pastagem_boa, dtype: float64


In [64]:
df[df['percent_area_pastagem'] > 0.8]

,V010100,NUM_QUADRA,NUM_FACE,V010800,area_total_estabel,area_pastagem_natural,area_pastagem_plantada,area_pastagem_degradada,area_pastagem_total,percent_area_pastagem,percent_area_pastagem_boa
20,291835705000014,0,233,233,58.806000,NaN,15.246000,38.333,53.578999,0.911114,NaN
26,290115505000011,0,117,117,130.679993,NaN,117.176003,NaN,117.176003,0.896664,NaN
33,171660405000018,0,30,29,45.980000,NaN,41.624001,NaN,41.624001,0.905263,NaN
35,292440510000014,0,237,236,30.000000,24.799999,NaN,NaN,24.799999,0.826667,NaN
54,292370405000036,0,97,97,50.000000,30.000000,NaN,15.000,45.000000,0.900000,NaN
...,...,...,...,...,...,...,...,...,...,...,...
5474,110149205000009,0,118,118,61.709999,NaN,55.660000,NaN,55.660000,0.901961,NaN
5477,520800410000004,0,265,266,30.000000,29.000000,NaN,NaN,29.000000,0.966667,NaN
5479,170220805000030,0,197,192,30.000000,NaN,29.000000,NaN,29.000000,0.966667,NaN
5496,110006405000041,0,38,38,61.000000,NaN,50.000000,NaN,50.000000,0.819672,NaN


In [65]:
df[df['percent_area_pastagem_boa'] > 0.8]

,V010100,NUM_QUADRA,NUM_FACE,V010800,area_total_estabel,area_pastagem_natural,area_pastagem_plantada,area_pastagem_degradada,area_pastagem_total,percent_area_pastagem,percent_area_pastagem_boa
62,150070105000009,0,90,89,100.000000,50.000000,50.000000,NaN,100.000000,1.000000,1.000000
138,510770120000006,0,42,42,70.000000,9.900000,50.000000,NaN,59.900002,0.855714,0.855714
266,520670105000006,0,123,123,164.559998,35.816002,96.800003,NaN,132.615997,0.805882,0.805882
362,431730115000002,0,24,24,270.000000,249.000000,20.000000,NaN,269.000000,0.996296,0.996296
422,520310405000008,0,93,93,48.400002,9.196000,38.720001,NaN,47.916000,0.990000,0.990000
...,...,...,...,...,...,...,...,...,...,...,...
5214,171875805000017,0,30,30,246.839996,234.740005,4.840000,NaN,239.580002,0.970588,0.970588
5245,291690605000017,0,163,163,62.000000,49.000000,10.000000,NaN,59.000000,0.951613,0.951613
5287,210200205000027,0,18,18,137.000000,85.000000,46.000000,NaN,131.000000,0.956204,0.956204
5288,510770120000006,0,133,133,40.000000,20.000000,15.000000,NaN,35.000000,0.875000,0.875000


In [66]:
query_r17 = """
SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800,
       e.VW01170300 AS area_total_estabel,
       e.VW04050000 AS area_pastagem_natural,
       e.VW04060000 AS area_pastagem_plantada,
       e.VW04170000 AS area_pastagem_total
FROM estabel e
LEFT JOIN pecu p
  ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA
 AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
WHERE (e.VW04050000 > 0 OR e.VW04060000 > 0)
  AND (p.V14010000 = '1' OR p.V14010000 IS NULL)
  AND (p.V15010000 = '1' OR p.V15010000 IS NULL)
  AND (p.V20010000 = '1' OR p.V20010000 IS NULL)
  AND (p.V21010000 = '1' OR p.V21010000 IS NULL)
  AND (p.V18010000 = '1' OR p.V18010000 IS NULL)
  AND (p.V16010000 = '1' OR p.V16010000 IS NULL)
  AND (p.V17010000 = '1' OR p.V17010000 IS NULL)
"""

def marcar_r17(query_r17: str):
    with sqlite3.connect(DB_MAIN) as conn:
        df_r17 = pd.read_sql_query(query_r17, conn)

    if not df_r17.empty:
        df_r17["percent_area_pastagem"] = df_r17["area_pastagem_total"] / df_r17["area_total_estabel"]
        df_r17["percent_area_pastagem_boa"] = (df_r17["area_pastagem_natural"] + df_r17["area_pastagem_plantada"]) / df_r17["area_total_estabel"]
        df_r17 = df_r17.loc[
            (df_r17["percent_area_pastagem"] > 0.8) | (df_r17["percent_area_pastagem_boa"] > 0.8),
            key_cols
        ]
    else:
        df_r17 = pd.DataFrame(columns=key_cols)

    aplicar_regra_df("R17", df_r17)

## Regra 18

In [67]:
query = """
SELECT 
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800,
    e.VW01170300 as area_total_estabel,
    e.VW04050000 as area_pastagem_natural
FROM estabel e 
LEFT JOIN pecu p ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
WHERE 
    (e.VW04050000 > 0 OR e.VW04060000 > 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df = pd.read_sql_query(query, conn)

df["percent_area_pastagem"] = df["area_pastagem_natural"] / df["area_total_estabel"]
df["percent_area_pastagem"].describe()

df[df['percent_area_pastagem'] > 0.9]

,V010100,NUM_QUADRA,NUM_FACE,V010800,area_total_estabel,area_pastagem_natural,percent_area_pastagem
25,521405105000008,0,28,28,1936.000000,1911.800049,0.987500
27,431830910000002,0,27,27,115.000000,113.800003,0.989565
44,520960605000003,0,125,124,43.560001,41.139999,0.944444
134,292335705000022,0,166,166,43.560001,43.341999,0.994995
156,292420705000015,0,2,2,32.973000,32.368000,0.981652
...,...,...,...,...,...,...,...
81896,330420105000138,0,24,24,290.399994,266.200012,0.916667
81897,292440510000007,0,17,17,69.695999,67.387001,0.966870
81911,315340005000023,0,53,53,30.000000,28.000000,0.933333
81938,260260505000035,0,122,122,303.000000,300.000000,0.990099


In [68]:
query_r18 = """
SELECT 
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800,
    e.VW01170300 as area_total_estabel,
    e.VW04050000 as area_pastagem_natural
FROM estabel e 
LEFT JOIN pecu p ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
WHERE 
    (e.VW04050000 > 0 OR e.VW04060000 > 0)
    AND (p.V14010000 = '1' OR p.V14010000 IS NULL)
    AND (p.V15010000 = '1' OR p.V15010000 IS NULL)
    AND (p.V20010000 = '1' OR p.V20010000 IS NULL)
    AND (p.V21010000 = '1' OR p.V21010000 IS NULL)
    AND (p.V18010000 = '1' OR p.V18010000 IS NULL)
    AND (p.V16010000 = '1' OR p.V16010000 IS NULL)
    AND (p.V17010000 = '1' OR p.V17010000 IS NULL);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df = pd.read_sql_query(query_r18, conn)

df["percent_area_pastagem"] = df["area_pastagem_natural"] / df["area_total_estabel"]
df["percent_area_pastagem"].describe()

df[df['percent_area_pastagem'] > 0.9]

,V010100,NUM_QUADRA,NUM_FACE,V010800,area_total_estabel,area_pastagem_natural,percent_area_pastagem
80,421700605000035,0,15,15,2.000000,2.000000,1.000000
165,220950005000006,0,158,155,80.000000,78.000000,0.975000
194,330060505000070,0,13,13,9.680000,9.583000,0.989979
211,260070805000018,0,42,42,0.034000,0.034000,1.000000
241,231310410000007,0,3,3,35.000000,32.990002,0.942571
...,...,...,...,...,...,...,...
5388,520800410000004,0,307,307,33.000000,32.000000,0.969697
5415,270460905000008,0,7,7,60.500000,57.173000,0.945008
5420,260620010000028,0,18,18,0.010000,0.010000,1.000000
5456,251100405000004,0,234,230,184.899994,178.899994,0.967550


In [69]:
def marcar_r18(query_r18: str):
    with sqlite3.connect(DB_MAIN) as conn:
        df_r18 = pd.read_sql_query(query_r18, conn)

    if not df_r18.empty:
        df_r18["percent_area_pastagem"] = df_r18["area_pastagem_natural"] / df_r18["area_total_estabel"]
        df_r18 = df_r18.loc[df_r18["percent_area_pastagem"] > 0.9, key_cols]
    else:
        df_r18 = pd.DataFrame(columns=key_cols)

    aplicar_regra_df("R18", df_r18)

## Regra 19

### Caso 1 - Área de pastagem sem despesas

In [70]:
query = """
SELECT COUNT(DISTINCT V010100 || NUM_QUADRA || NUM_FACE || V010800) as total_estabelecimentos
FROM estabel
WHERE V04060000 > 0 
AND (V45012600 = 0 OR V45012600 IS NULL);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 48032 -> 48.03%


### Caso 2 - Área de pastagem sem rebanho

In [71]:
query = """
SELECT 
    COUNT(DISTINCT e.V010100 || e.NUM_QUADRA || e.NUM_FACE || e.V010800) as total
FROM estabel e 
LEFT JOIN pecu p ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
WHERE 
    (e.V04050000 > 0 OR e.V04060000 > 0)
    AND (p.V14010000 = '1' OR p.V14010000 IS NULL)
    AND (p.V15010000 = '1' OR p.V15010000 IS NULL)
    AND (p.V20010000 = '1' OR p.V20010000 IS NULL)
    AND (p.V21010000 = '1' OR p.V21010000 IS NULL)
    AND (p.V18010000 = '1' OR p.V18010000 IS NULL)
    AND (p.V16010000 = '1' OR p.V16010000 IS NULL)
    AND (p.V17010000 = '1' OR p.V17010000 IS NULL);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 5504 -> 5.50%


In [72]:
query_r19 = """
SELECT 
    DISTINCT V010100, NUM_QUADRA, NUM_FACE, V010800
FROM (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM estabel
    WHERE V04060000 > 0 AND (V45012600 = 0 OR V45012600 IS NULL)

    UNION

    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM estabel e
    LEFT JOIN pecu p
      ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA
     AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
    WHERE (e.V04050000 > 0 OR e.V04060000 > 0)
      AND (p.V14010000 = '1' OR p.V14010000 IS NULL)
      AND (p.V15010000 = '1' OR p.V15010000 IS NULL)
      AND (p.V20010000 = '1' OR p.V20010000 IS NULL)
      AND (p.V21010000 = '1' OR p.V21010000 IS NULL)
      AND (p.V18010000 = '1' OR p.V18010000 IS NULL)
      AND (p.V16010000 = '1' OR p.V16010000 IS NULL)
      AND (p.V17010000 = '1' OR p.V17010000 IS NULL)
) t
"""

## Regra 20

In [73]:
query_r20 = None

## Regra 21

In [74]:
query = """
SELECT
    COUNT(*) as total
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica = 345
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 308 -> 0.31%


In [75]:
query_r21 = """
SELECT 
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
FROM estabel e
JOIN estabel_criticas ec
  ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE ec.id_critica = 345
"""

## Regra 22

In [76]:
query = """
SELECT 
    COUNT(DISTINCT e.V010100 || e.NUM_QUADRA || e.NUM_FACE || e.V010800) AS total
FROM estabel e
LEFT JOIN efet_silv es
    ON e.V010100 = es.V010100
   AND e.NUM_QUADRA = es.NUM_QUADRA
   AND e.NUM_FACE = es.NUM_FACE
   AND e.V010800 = es.V010800
LEFT JOIN silv s
    ON e.V010100 = s.V010100
   AND e.NUM_QUADRA = s.NUM_QUADRA
   AND e.NUM_FACE = s.NUM_FACE
   AND e.V010800 = s.V010800
WHERE 
    e.V04100000 > 0 AND e.V45012500 > 0
    AND (es.V39010100 IS NULL OR es.V39020300 = 0)
    AND (s.V40010100 IS NULL OR s.V40020300 = 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 29 -> 0.03%


In [77]:
query_r22 = """
SELECT 
    DISTINCT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
FROM estabel e
LEFT JOIN efet_silv es
  ON e.V010100 = es.V010100 AND e.NUM_QUADRA = es.NUM_QUADRA AND e.NUM_FACE = es.NUM_FACE AND e.V010800 = es.V010800
LEFT JOIN silv s
  ON e.V010100 = s.V010100 AND e.NUM_QUADRA = s.NUM_QUADRA AND e.NUM_FACE = s.NUM_FACE AND e.V010800 = s.V010800
WHERE e.V04100000 > 0
  AND e.V45012500 > 0
  AND (es.V39010100 IS NULL OR es.V39020300 = 0)
  AND (s.V40010100 IS NULL OR s.V40020300 = 0)
"""

## Regra 23

In [78]:
query = """
SELECT
    V010100, NUM_QUADRA, NUM_FACE, V010800,
    VW01170300 as area_total_estabel,
    VW04080000 as area_protecao_permanente,
    VW04120000 as area_lamina_dagua
FROM estabel"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df = pd.read_sql_query(query, conn)

df['percent_lamina_dagua'] = df['area_lamina_dagua'] / df['area_total_estabel']
display(df['percent_lamina_dagua'].describe())

df['percent_protecao_permanente'] = df['area_protecao_permanente'] / df['area_total_estabel']
display(df['percent_protecao_permanente'].describe())

count    90814.000000
mean         0.056255
std          0.119932
min          0.000000
25%          0.008427
50%          0.020833
75%          0.050000
max          1.000000
Name: percent_lamina_dagua, dtype: float64

count    68468.000000
mean         0.244200
std          0.203881
min          0.000000
25%          0.101695
50%          0.200000
75%          0.300000
max          1.000000
Name: percent_protecao_permanente, dtype: float64

In [79]:
df[df['percent_lamina_dagua'] > 0.95]

,V010100,NUM_QUADRA,NUM_FACE,V010800,area_total_estabel,area_protecao_permanente,area_lamina_dagua,percent_lamina_dagua,percent_protecao_permanente
170,320190205000008,0,151,147,0.725000,NaN,0.725000,1.000000,NaN
353,150680705000152,0,110,106,32.000000,NaN,31.875000,0.996094,NaN
929,230470705000016,0,28,28,56.000000,0.15,55.299999,0.987500,0.002679
982,210005505000121,0,174,174,82.279999,NaN,79.289001,0.963649,NaN
1657,150170920000002,0,1,1,100.000000,NaN,99.750000,0.997500,NaN
...,...,...,...,...,...,...,...,...,...
99209,320455905000051,0,116,114,0.070000,NaN,0.070000,1.000000,NaN
99685,411790905000029,0,42,42,1.000000,NaN,1.000000,1.000000,NaN
99698,230540705000036,0,95,94,84.000000,NaN,83.000000,0.988095,NaN
99899,292890110000003,0,12,12,250.000000,NaN,245.000000,0.980000,NaN


In [80]:
df[df['percent_protecao_permanente'] > 0.95]

,V010100,NUM_QUADRA,NUM_FACE,V010800,area_total_estabel,area_protecao_permanente,area_lamina_dagua,percent_lamina_dagua,percent_protecao_permanente
523,290475305000025,0,14,14,45.000000,43.000000,1.000,0.022222,0.955556
558,130070605000037,0,16,16,200.000000,197.600006,0.400,0.002000,0.988000
593,150495005000015,2,12,20011,50.215000,49.005001,NaN,NaN,0.975904
611,130150605000043,0,33,33,240.000000,236.300003,0.500,0.002083,0.984583
940,120060905000078,0,115,115,100.000000,99.000000,0.500,0.005000,0.990000
...,...,...,...,...,...,...,...,...,...
98721,290750910000004,0,137,137,108.900002,103.672997,0.436,0.004004,0.952002
98788,293020430000004,0,108,108,52.000000,52.000000,NaN,NaN,1.000000
98899,150060205000140,0,339,339,48.400002,47.431999,0.484,0.010000,0.980000
99026,221063105000008,0,19,19,70.000000,67.500000,0.500,0.007143,0.964286


In [81]:
query_r23 = """
SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800,
    VW01170300 AS area_total_estabel,
    VW04080000 AS area_protecao_permanente,
    VW04120000 AS area_lamina_dagua
FROM estabel
"""

def marcar_r23(query_r23: str):
    with sqlite3.connect(DB_MAIN) as conn:
        df_r23 = pd.read_sql_query(query_r23, conn)

    if not df_r23.empty:
        df_r23["percent_lamina_dagua"] = df_r23["area_lamina_dagua"] / df_r23["area_total_estabel"]
        df_r23["percent_protecao_permanente"] = df_r23["area_protecao_permanente"] / df_r23["area_total_estabel"]
        df_r23 = df_r23.loc[
            (df_r23["percent_lamina_dagua"] > 0.95)
            | (df_r23["percent_protecao_permanente"] > 0.95),
            key_cols
        ]
    else:
        df_r23 = pd.DataFrame(columns=key_cols)

    aplicar_regra_df("R23", df_r23)

## Regra 24

In [82]:
query = """SELECT COUNT(*) FROM estabel WHERE VW05270104 > VW01170300"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 39 -> 0.04%


In [83]:
query_r24 = """
SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800
FROM estabel
WHERE VW05270104 > VW01170300
"""

## Regra 25

In [84]:
query = """
SELECT 
    COUNT(*) 
FROM estabel 
WHERE 
    V05270103 = '2' 
    AND VW05270104 > 0 
    AND (V05250100 = '2' OR V05250100 = '4')
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 76 -> 0.08%


In [85]:
query = """
SELECT 
    COUNT(*) 
FROM estabel 
WHERE 
    V05270103 = '2' 
    AND VW05270104 > 0 
    AND V05180100 = '1'
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 2085 -> 2.08%


In [86]:
query = """
SELECT 
    COUNT(*) 
FROM estabel 
WHERE 
    V05270103 = '2' 
    AND VW05270104 > 0 
    AND V45010800 = 0 OR V45010800 IS NULL
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 63244 -> 63.24%


In [87]:
query_r25 = """
SELECT 
    V010100, NUM_QUADRA, NUM_FACE, V010800
FROM estabel
WHERE
    (
        V05270103 = '2'
        AND VW05270104 > 0
        AND (V05250100 = '2' OR V05250100 = '4')
    )
"""

# OR
#     (
#         V05270103 = '2'
#         AND VW05270104 > 0
#         AND (V45010800 = 0 OR V45010800 IS NULL)
#     )

## Regra 26

### Caso 1 - Área menor que 10 hectares

In [88]:
query = """SELECT COUNT(*) FROM estabel WHERE VW05270104 < 10"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 4071 -> 4.07%


### Caso 2 - Área extensa sem máquinarios

In [89]:
query = """
SELECT COUNT(*) as total
FROM estabel
WHERE VW05270104 > 100 
AND (V07020100 = '1' or V07020100 IS NULL)
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 298 -> 0.30%


In [90]:
query = """
SELECT COUNT(*) as total
FROM estabel
WHERE 
    VW05270104 > 100 
    AND (V07020100 = '1' or V07020100 IS NULL)
    AND (VW10040101 = 0 OR VW10040101 IS NULL)
    AND (VW10040201 = 0 OR VW10040201 IS NULL)
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 128 -> 0.13%


In [91]:
query_r26 = """
SELECT 
    DISTINCT V010100, NUM_QUADRA, NUM_FACE, V010800
FROM (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM estabel
    WHERE VW05270104 < 10

    UNION

    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM estabel e
    JOIN pecu p
      ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA
     AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
    WHERE VW05270104 > 100
      AND (V07020100 = '1' OR V07020100 IS NULL)
      AND (VW10040101 = 0 OR VW10040101 IS NULL)
      AND (VW10040201 = 0 OR VW10040201 IS NULL)
) t
"""

## Regra 27

In [92]:
query = """
SELECT
    COUNT(*) as total
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica = 67
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 113 -> 0.11%


In [93]:
query = """
SELECT
    COUNT(*) as total
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica = 68
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 8 -> 0.01%


In [94]:
query_r27 = """
SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
FROM estabel e
JOIN estabel_criticas ec
  ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE ec.id_critica IN (67, 68)
"""

## Regra 28

In [95]:
query = """
SELECT 
    COUNT(*) as total
FROM estabel
WHERE 
    V05180100 = '2' 
    AND (V45010800 IS NULL OR V45010800 = 0);
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 572 -> 0.57%


In [96]:
query = """
SELECT
    COUNT(*) as total
FROM estabel e
JOIN estabel_criticas ec ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
WHERE
    ec.id_critica = 377
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 572 -> 0.57%


In [97]:
query_r28 = """
SELECT DISTINCT V010100, NUM_QUADRA, NUM_FACE, V010800
FROM (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
    FROM estabel
    WHERE V05180100 = '2' AND (V45010800 IS NULL OR V45010800 = 0)

    UNION

    SELECT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
    FROM estabel e
    JOIN estabel_criticas ec
      ON e.V010100 = ec.V010100 AND e.NUM_QUADRA = ec.NUM_QUADRA AND e.NUM_FACE = ec.NUM_FACE AND e.V010800 = ec.V010800
    WHERE ec.id_critica = 377
) t
"""

## Regra 29

In [98]:
query = """
SELECT 
    COUNT(DISTINCT e.V010100 || e.NUM_QUADRA || e.NUM_FACE || e.V010800)
FROM estabel e
JOIN lav_temp l ON e.V010100 = l.V010100 AND e.NUM_QUADRA = l.NUM_QUADRA AND e.NUM_FACE = l.NUM_FACE AND e.V010800 = l.V010800
WHERE 
    (l.V34010100 IN (262, 263, 266, 264)) 
    AND e.V13080000 = '1'
"""

# 262 -> Cana Forrageira
# 263 -> Milho Forrageiro
# 264 -> Sorgo Forrageiro
# 266 -> Palma Forrageira

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 11178 -> 11.18%


In [99]:
# V13010000

query = """
SELECT 
    COUNT(DISTINCT e.V010100 || e.NUM_QUADRA || e.NUM_FACE || e.V010800)
FROM estabel e
LEFT JOIN lav_temp l ON e.V010100 = l.V010100 AND e.NUM_QUADRA = l.NUM_QUADRA AND e.NUM_FACE = l.NUM_FACE AND e.V010800 = l.V010800
WHERE 
    (l.V34010100 IN (262, 263, 266, 264)) 
    AND e.V13010000 = '1'
"""

# 262 -> Cana Forrageira
# 263 -> Milho Forrageiro
# 264 -> Sorgo Forrageiro
# 266 -> Palma Forrageira

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 139 -> 0.14%


In [100]:
# query_r29 = """
# SELECT DISTINCT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
# FROM estabel e
# LEFT JOIN lav_temp l
#   ON e.V010100 = l.V010100 AND e.NUM_QUADRA = l.NUM_QUADRA
#  AND e.NUM_FACE = l.NUM_FACE AND e.V010800 = l.V010800
# WHERE l.V34010100 IN (262, 263, 264, 266)
#   AND (e.V13080000 = '1' OR e.V13010000 = '1')
# """

query_r29 = """
SELECT DISTINCT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
FROM estabel e
LEFT JOIN lav_temp l
  ON e.V010100 = l.V010100 AND e.NUM_QUADRA = l.NUM_QUADRA
 AND e.NUM_FACE = l.NUM_FACE AND e.V010800 = l.V010800
WHERE l.V34010100 IN (262, 263, 264, 266)
  AND e.V13010000 = '1'
"""

## Regra 30

In [101]:
query = """
SELECT 
    COUNT(*) 
FROM estabel 
WHERE 
    V05270101 = '2' 
    AND (
        V33010100 = '1' 
        OR (V33010200 = '1' OR VW04020000 < 100)
    )
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 21783 -> 21.78%


In [102]:
query = """
SELECT 
    COUNT(*) 
FROM estabel 
WHERE 
    V05270101 = '2' 
    AND (
        V33010100 = '1' 
        OR (VW04030000 = 0 OR VW04030000 IS NULL)
    )
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 8016 -> 8.02%


In [103]:
query = """
SELECT 
    COUNT(*) 
FROM estabel 
WHERE 
    V05270101 = '2' 
    AND (
       VW04020000 < 100 OR VW04020000 IS NULL
    )
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 22090 -> 22.09%


In [104]:
query_r30 = """
SELECT V010100, NUM_QUADRA, NUM_FACE, V010800
FROM estabel
WHERE
    V05270101 = '2'
    AND (
        V33010100 = '1'
        OR (V33010200 = '1' OR VW04020000 < 100)
    )
"""

## Regra 31

In [105]:
query = """
SELECT 
    COUNT(DISTINCT e.V010100 || e.NUM_QUADRA || e.NUM_FACE || e.V010800) as total
FROM estabel e
LEFT JOIN pecu p ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
WHERE 
    e.V05270101 = '2' 
    AND e.V07020100 = '1' 
    AND e.V13011600 = '1' 
    AND e.V13011700 = '1' 
    AND e.V13011800 = '1'
    AND e.V13011500 = '1'
    AND (e.V13011400 = '1' OR p.V14020101 != '6')
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute(query)
    count = cursor.fetchone()[0]
    print(f"Total de registros: {count} -> {count/QTD_ESTABELECIMENTOS:.2%}")

Total de registros: 852 -> 0.85%


In [106]:
query_r31 = """
SELECT 
    DISTINCT e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800
FROM estabel e
LEFT JOIN pecu p
  ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA
 AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
WHERE e.V05270101 = '2'
  AND e.V07020100 = '1'
  AND e.V13011600 = '1'
  AND e.V13011700 = '1'
  AND e.V13011800 = '1'
  AND e.V13011500 = '1'
  AND (e.V13011400 = '1' OR p.V14020101 != '6')
"""

## Regra 32

In [107]:
# V02090000 - estabel - finalidade_producao_agropecu
# V32020100 - pecu - objetivo_principal_pesca

query = """
WITH
agro_ind_agg AS (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800, COALESCE(SUM(VW41030501), 0) AS valor_venda_agroindustria
    FROM agro_ind
    GROUP BY V010100, NUM_QUADRA, NUM_FACE, V010800
),
extr_veg_agg AS (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800, COALESCE(SUM(VW36020401), 0) AS valor_venda_extracao_vegetal
    FROM extr_veg
    GROUP BY V010100, NUM_QUADRA, NUM_FACE, V010800
),
flori_agg AS (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800, COALESCE(SUM(V38020401), 0) AS valor_venda_floricultura
    FROM flori
    GROUP BY V010100, NUM_QUADRA, NUM_FACE, V010800
),
hort_agg AS (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800, COALESCE(SUM(VW37020401), 0) AS valor_venda_horticultura
    FROM hort
    GROUP BY V010100, NUM_QUADRA, NUM_FACE, V010800
),
lav_per_agg AS (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800, COALESCE(SUM(VW35020400), 0) AS valor_venda_lavoura_permanente
    FROM lav_per
    GROUP BY V010100, NUM_QUADRA, NUM_FACE, V010800
),
lav_temp_agg AS (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800, COALESCE(SUM(VW34020902), 0) AS valor_venda_lavoura_temporaria
    FROM lav_temp
    GROUP BY V010100, NUM_QUADRA, NUM_FACE, V010800
),
silv_agg AS (
    SELECT V010100, NUM_QUADRA, NUM_FACE, V010800, COALESCE(SUM(VW40020301), 0) AS valor_venda_silvicultura
    FROM silv
    GROUP BY V010100, NUM_QUADRA, NUM_FACE, V010800
)

SELECT
    e.V010100, e.NUM_QUADRA, e.NUM_FACE, e.V010800,

    COALESCE(ai.valor_venda_agroindustria, 0) AS valor_venda_agroindustria,
    COALESCE(ev.valor_venda_extracao_vegetal, 0) AS valor_venda_extracao_vegetal,
    COALESCE(fl.valor_venda_floricultura, 0) AS valor_venda_floricultura,
    COALESCE(ho.valor_venda_horticultura, 0) AS valor_venda_horticultura,
    COALESCE(lp.valor_venda_lavoura_permanente, 0) AS valor_venda_lavoura_permanente,
    COALESCE(lt.valor_venda_lavoura_temporaria, 0) AS valor_venda_lavoura_temporaria,
    COALESCE(si.valor_venda_silvicultura, 0) AS valor_venda_silvicultura,

    COALESCE(p.V14130102, 0) AS valor_venda_bovinos,
    COALESCE(p.V14130202, 0) AS valor_venda_reprodutores,
    COALESCE(p.V14130302, 0) AS valor_venda_cria_engorda,
    COALESCE(p.V14130402, 0) AS valor_venda_abate,
    COALESCE(p.VW14160300, 0) AS valor_venda_leite_bovino,
    COALESCE(p.V15010502, 0) AS valor_bubalinos,
    COALESCE(p.VW15020300, 0) AS valor_venda_leite_bubalino,
    COALESCE(p.V16010502, 0) AS valor_equinos,
    COALESCE(p.V17010502, 0) AS valor_venda_asininos,
    COALESCE(p.V18010502, 0) AS valor_venda_muares,
    COALESCE(p.V19030402, 0) AS valor_venda_suinos,
    COALESCE(p.V20030402, 0) AS valor_venda_caprinos,
    COALESCE(p.VW20040301, 0) AS valor_venda_leite_caprino,
    COALESCE(p.V21030402, 0) AS valor_venda_ovinos,
    COALESCE(p.V21050301, 0) AS valor_venda_leite_ovelha,
    COALESCE(p.V22050002, 0) AS valor_venda_aves,
    COALESCE(p.VW22080400, 0) AS valor_venda_ovos,
    COALESCE(p.VW25030000, 0) AS valor_venda_codornas,
    COALESCE(p.VW25040200, 0) AS valor_venda_ovos_codornas,
    COALESCE(p.V26030102, 0) AS valor_venda_outras_aves,
    COALESCE(p.V26040202, 0) AS valor_venda_ovos_outras_aves,
    COALESCE(p.V27010302, 0) AS valor_venda_coelhos,
    COALESCE(p.VW28020200, 0) AS valor_venda_mel_abelhas,
    COALESCE(p.VW28030200, 0) AS valor_venda_cera_abelhas,
    COALESCE(p.VW29050201, 0) AS valor_venda_peixes,
    COALESCE(p.VW29060201, 0) AS valor_venda_camarao,
    COALESCE(p.VW29070201, 0) AS valor_venda_ostras,
    COALESCE(p.VW29080201, 0) AS valor_venda_mexilhoes,
    COALESCE(p.VW30010200, 0) AS valor_venda_carne_ra,
    COALESCE(p.VW31010200, 0) AS valor_venda_casulos_bicho_da_seda,

    e.V46010700 as renda_fora_estabelecimento

FROM estabel e
LEFT JOIN pecu p
    ON e.V010100 = p.V010100 AND e.NUM_QUADRA = p.NUM_QUADRA AND e.NUM_FACE = p.NUM_FACE AND e.V010800 = p.V010800
LEFT JOIN agro_ind_agg ai
    ON e.V010100 = ai.V010100 AND e.NUM_QUADRA = ai.NUM_QUADRA AND e.NUM_FACE = ai.NUM_FACE AND e.V010800 = ai.V010800
LEFT JOIN extr_veg_agg ev
    ON e.V010100 = ev.V010100 AND e.NUM_QUADRA = ev.NUM_QUADRA AND e.NUM_FACE = ev.NUM_FACE AND e.V010800 = ev.V010800
LEFT JOIN flori_agg fl
    ON e.V010100 = fl.V010100 AND e.NUM_QUADRA = fl.NUM_QUADRA AND e.NUM_FACE = fl.NUM_FACE AND e.V010800 = fl.V010800
LEFT JOIN hort_agg ho
    ON e.V010100 = ho.V010100 AND e.NUM_QUADRA = ho.NUM_QUADRA AND e.NUM_FACE = ho.NUM_FACE AND e.V010800 = ho.V010800
LEFT JOIN lav_per_agg lp
    ON e.V010100 = lp.V010100 AND e.NUM_QUADRA = lp.NUM_QUADRA AND e.NUM_FACE = lp.NUM_FACE AND e.V010800 = lp.V010800
LEFT JOIN lav_temp_agg lt
    ON e.V010100 = lt.V010100 AND e.NUM_QUADRA = lt.NUM_QUADRA AND e.NUM_FACE = lt.NUM_FACE AND e.V010800 = lt.V010800
LEFT JOIN silv_agg si
    ON e.V010100 = si.V010100 AND e.NUM_QUADRA = si.NUM_QUADRA AND e.NUM_FACE = si.NUM_FACE AND e.V010800 = si.V010800
"""

with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    df_vendas = pd.read_sql_query(query, conn)

df_vendas

,V010100,NUM_QUADRA,NUM_FACE,V010800,valor_venda_agroindustria,valor_venda_extracao_vegetal,valor_venda_floricultura,valor_venda_horticultura,valor_venda_lavoura_permanente,valor_venda_lavoura_temporaria,...,valor_venda_coelhos,valor_venda_mel_abelhas,valor_venda_cera_abelhas,valor_venda_peixes,valor_venda_camarao,valor_venda_ostras,valor_venda_mexilhoes,valor_venda_carne_ra,valor_venda_casulos_bicho_da_seda,renda_fora_estabelecimento
0,421050605000032,0,97,97,0.0,0.0,0.0,0.0,0.0,382750.0,...,0,0.0,0,0.0,0,0,0,0,0,16800.0
1,510618205000023,0,151,150,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0,0,0,0,NaN
2,250800005000010,0,146,146,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0,0,0,0,NaN
3,170200005000011,0,11,11,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0,0,0,0,NaN
4,313330305000021,0,64,64,11520.0,0.0,0.0,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0,0,0,0,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,510677805000013,0,11,11,9900.0,0.0,0.0,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0,0,0,0,NaN
99996,140017505000011,0,102,102,0.0,0.0,0.0,0.0,0.0,180.0,...,0,0.0,0,0.0,0,0,0,0,0,NaN
99997,432110505000030,0,13,13,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0,0,0,0,NaN
99998,170300805000007,0,35,35,0.0,0.0,0.0,0.0,0.0,0.0,...,0,0.0,0,0.0,0,0,0,0,0,NaN


In [108]:
query_r32 = None

## Regra Especial

In [109]:
with sqlite3.connect("../data/amstr_geral_criticas.db") as conn:
    cursor = conn.cursor()
    cursor.execute("SELECT count(*) FROM estabel WHERE V04020001=2 AND ((V04020000 IS NULL OR V04020000 = 0) AND (V04030000 IS NULL OR V04030000 = 0) AND (V04040000 IS NULL OR V04040000 = 0));")
    count = cursor.fetchone()[0]
    print(count)

4


# Consolidacao da marcacao das regras

Esta secao garante que todas as regras tenham coluna booleana em `df_estabelecimentos`.

- Quando a regra pode ser representada por SQL de chaves, usamos `compilar_regra`.
- Quando depende de percentil ou tratamento em pandas, fazemos a marcacao por merge em DataFrame.
- Regras sem definicao operacional no notebook (R01, R08, R09, R20 e R32) sao inicializadas com 0 ate definicao futura.

In [111]:
df_estabelecimentos = preparar_base_estabelecimentos()

sql_rules = [
    (1, query_r1, DB_CRIT),
    # (2, query_r2, DB_CRIT), # Ignorada para controlar o limiar de contaminação
    # (3, query_r3, DB_CRIT), # Ignorada por superar o limiar de contaminação
    (4, query_r4, DB_CRIT),
    # (5, query_r5, DB_CRIT), # Ignorada por superar o limiar de contaminação
    # (7, query_r7, DB_CRIT), # Ignorada para controlar o limiar de contaminação
    # (11, query_r11, DB_CRIT), # Ignorada por superar o limiar de contaminação
    (13, query_r13, DB_CRIT),
    # (14, query_r14, DB_CRIT), # Ignorada para controlar o limiar de contaminação
    (15, query_r15, DB_CRIT),
    # (19, query_r19, DB_CRIT), # Ignorada pois foi revogada após análise
    (21, query_r21, DB_CRIT),
    (22, query_r22, DB_CRIT),
    (24, query_r24, DB_CRIT),
    (25, query_r25, DB_CRIT),
    # (26, query_r26, DB_CRIT), # Ignorada por superar o limiar de contaminação
    (27, query_r27, DB_CRIT),
    (28, query_r28, DB_CRIT),
    (29, query_r29, DB_CRIT),
    # (30, query_r30, DB_CRIT), # Ignorada por superar o limiar de contaminação
    (31, query_r31, DB_CRIT),
]

for numero_regra, query_regra, db_path in sql_rules:
    aplicar_regra_sql(f"R{numero_regra:02d}", query_regra, db_path)

marcar_r6(query_r6)
# marcar_r10(query_r10) # Ignorada para controlar o limiar de contaminação
marcar_r12(query_r12)
marcar_r16(query_r16)
marcar_r17(query_r17)
marcar_r18(query_r18)
marcar_r23(query_r23)

for numero_regra in [1, 8, 9, 20, 32] + [10]:
    aplicar_regra_vazia(f"R{numero_regra:02d}")


# Garante estrutura final das colunas
regras_esperadas = [f"R{i:02d}" for i in range(1, 33)]
for regra in regras_esperadas:
    if regra not in df_estabelecimentos.columns:
        df_estabelecimentos[regra] = 0
    df_estabelecimentos[regra] = pd.to_numeric(
        df_estabelecimentos[regra], errors="coerce"
    ).fillna(0).astype("Int8")

resumo_regras = (
    df_estabelecimentos[regras_esperadas]
    .sum()
    .sort_index()
    .to_frame("qtd_marcados")
)

display(resumo_regras)
# display(df_estabelecimentos.head())

,qtd_marcados
R01,0
R02,0
R03,0
R04,4
R05,0
R06,92
R07,0
R08,0
R09,0
R10,0


In [112]:
colunas_regras = [col for col in df_estabelecimentos.columns if col.startswith("R")]
df_estabelecimentos["total_regras_marcadas"] = df_estabelecimentos[colunas_regras].sum(axis=1)
df_estabelecimentos["total_regras_marcadas"].describe()

count    100000.0
mean      0.07541
std       0.29073
min           0.0
25%           0.0
50%           0.0
75%           0.0
max           4.0
Name: total_regras_marcadas, dtype: Float64

In [113]:
df_estabelecimentos['caiu_em_alguma_regra'] = df_estabelecimentos[colunas_regras].any(axis=1).astype(int)
df_estabelecimentos['caiu_em_alguma_regra'].value_counts()

print("% de estabelecimentos que caíram em pelo menos uma regra:", round(df_estabelecimentos['caiu_em_alguma_regra'].mean() * 100, 2), "%")

% de estabelecimentos que caíram em pelo menos uma regra: 6.87 %


# Contaminação por Tabela

In [114]:
# Base de referência no notebook
df_ref = df_estabelecimentos.copy()

# Garante indicador de contaminação
if "caiu_em_alguma_regra" not in df_ref.columns:
    cols_regras = [c for c in df_ref.columns if c.startswith("R")]
    df_ref["caiu_em_alguma_regra"] = df_ref[cols_regras].any(axis=1).astype(int)

df_ref_keys = df_ref[key_cols].drop_duplicates()
df_contaminados = df_ref.loc[df_ref["caiu_em_alguma_regra"] == 1, key_cols].drop_duplicates()

total_df = len(df_ref_keys)
total_contaminados_df = len(df_contaminados)

with sqlite3.connect(DB_MAIN) as conn:
    # Tabelas elegíveis: exclui estabel e qualquer tabela relacionada a crítica
    tabelas = pd.read_sql_query(
        """
        SELECT name
        FROM sqlite_master
        WHERE type = 'table'
          AND name NOT LIKE 'sqlite_%'
        """,
        conn
    )["name"].tolist()

    tabelas = [
        t for t in tabelas
        if t.lower() != "estabel" and "critica" not in t.lower()
    ]

    # Salva contaminados como tabela temporária para joins SQL
    df_contaminados.to_sql("_tmp_contaminados", conn, if_exists="replace", index=False)

    resultados = []
    for tabela in tabelas:
        cols_tabela = pd.read_sql_query(f'PRAGMA table_info("{tabela}")', conn)["name"].tolist()
        if not set(key_cols).issubset(cols_tabela):
            continue

        sql_total = f"""
            SELECT COUNT(*) AS n
            FROM (
                SELECT DISTINCT {", ".join(key_cols)}
                FROM "{tabela}"
            ) t
        """

        sql_intersec = f"""
            SELECT COUNT(*) AS n
            FROM (
                SELECT DISTINCT {", ".join(key_cols)}
                FROM "{tabela}"
            ) t
            INNER JOIN _tmp_contaminados c
                ON {" AND ".join([f"t.{k} = c.{k}" for k in key_cols])}
        """

        total_tabela = pd.read_sql_query(sql_total, conn).iloc[0, 0]
        contaminados_na_tabela = pd.read_sql_query(sql_intersec, conn).iloc[0, 0]

        resultados.append({
            "tabela": tabela,
            "total_estabs_tabela": int(total_tabela),
            "contaminados_na_tabela": int(contaminados_na_tabela),
            "%_contaminacao_na_tabela": (contaminados_na_tabela / total_tabela * 100) if total_tabela else 0.0,
            "%_contaminacao_vs_df_total": (contaminados_na_tabela / total_df * 100) if total_df else 0.0,
            "%_contaminacao_vs_df_contaminado": (contaminados_na_tabela / total_contaminados_df * 100) if total_contaminados_df else 0.0,
        })

df_contaminacao_tabelas = pd.DataFrame(resultados).sort_values(
    by="%_contaminacao_vs_df_total", ascending=False
).reset_index(drop=True)

display(df_contaminacao_tabelas)

,tabela,total_estabs_tabela,contaminados_na_tabela,%_contaminacao_na_tabela,%_contaminacao_vs_df_total,%_contaminacao_vs_df_contaminado
0,pecu,91382,5994,6.559279,5.994,87.261610
1,lav_temp,48193,3474,7.208516,3.474,50.575047
2,lav_per,24199,1780,7.355676,1.780,25.913525
3,agro_ind,15319,1036,6.762844,1.036,15.082254
4,efet_silv,7735,686,8.868778,0.686,9.986898
5,extr_veg,7845,577,7.355003,0.577,8.400058
6,hort,2712,293,10.803835,0.293,4.265541
7,silv,2569,206,8.018684,0.206,2.998981
8,flori,106,9,8.490566,0.009,0.131023


In [115]:
queries_used = {
    "r04": query_r4,
    "r06": query_r6,
    "r12": query_r12,
    "r13": query_r13,
    "r15": query_r15,
    "r16": query_r16,
    "r17": query_r17,
    "r18": query_r18,
    "r21": query_r21,
    "r22": query_r22,
    "r23": query_r23,
    "r24": query_r24,
    "r25": query_r25,
    "r27": query_r27,
    "r28": query_r28,
    "r29": query_r29,
    "r31": query_r31,
}

tables_useds = {tabela: [] for tabela in tabelas + ["estabel"]}

for tabela in tabelas + ["estabel"]:
    for label, query in queries_used.items():
        label = label.upper()
        if isinstance(query, str) and tabela in query:
            tables_useds[tabela].append(label)
        elif isinstance(query, list):
            for subquery in query:
                if tabela in subquery:
                    tables_useds[tabela].append(label)
        elif isinstance(query, dict):
            for key, subquery in query.items():
                if tabela in subquery:
                    tables_useds[tabela].append(label)
    
for tabela, labels in tables_useds.items():
    if not labels:
        print(f"Contaminação da tabela {tabela} = 0% (não utilizada em nenhuma regra)")
        continue
    
    df_table_set = df_estabelecimentos.loc[:, key_cols + labels]
    df_table_set["caiu_em_alguma_regra"] = df_table_set[labels].any(axis=1).astype(int)
    contaminacao = df_table_set["caiu_em_alguma_regra"].mean() * 100
    print(f"Contaminação da tabela {tabela} = {contaminacao:.2f}% (utilizada nas regras {', '.join(labels)})")

Contaminação da tabela agro_ind = 0% (não utilizada em nenhuma regra)
Contaminação da tabela efet_silv = 0.03% (utilizada nas regras R22)
Contaminação da tabela extr_veg = 0% (não utilizada em nenhuma regra)
Contaminação da tabela flori = 0% (não utilizada em nenhuma regra)
Contaminação da tabela hort = 0% (não utilizada em nenhuma regra)
Contaminação da tabela lav_per = 0.09% (utilizada nas regras R06)
Contaminação da tabela lav_temp = 1.04% (utilizada nas regras R06, R16, R29)
Contaminação da tabela pecu = 4.10% (utilizada nas regras R12, R12, R13, R15, R17, R18, R31)
Contaminação da tabela silv = 0.03% (utilizada nas regras R22)
Contaminação da tabela estabel = 6.87% (utilizada nas regras R04, R06, R06, R12, R13, R15, R16, R16, R17, R18, R21, R22, R23, R24, R25, R27, R28, R29, R31)


# Extraindo os Resultados das Regras como Ground Truth

In [118]:
df_estabelecimentos

,V010100,NUM_QUADRA,NUM_FACE,V010800,R04,R13,R15,R21,R22,R24,...,R03,R05,R07,R11,R14,R19,R26,R30,total_regras_marcadas,caiu_em_alguma_regra
0,421050605000032,0,97,97,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,510618205000023,0,151,150,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,250800005000010,0,146,146,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,170200005000011,0,11,11,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,313330305000021,0,64,64,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,510677805000013,0,11,11,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
99996,140017505000011,0,102,102,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
99997,432110505000030,0,13,13,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
99998,170300805000007,0,35,35,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [119]:
df_ground_truth = df_estabelecimentos.copy()

df_ground_truth['is_anomaly'] = df_ground_truth['caiu_em_alguma_regra']
df_ground_truth.drop('caiu_em_alguma_regra', axis=1, inplace=True)

df_ground_truth.to_csv("../data/experimentos/ground_truth.csv", index=False)

# Seleção da Base Para Validação

In [ ]:
# seed = 42
# n_total = 60
# n_sem_marcacao_min = 15

# regras_nao_zeradas = [r for r in colunas_regras if df_estabelecimentos[r].sum() > 0]
# mask_sem_marcacao = df_estabelecimentos[colunas_regras].sum(axis=1) == 0

# if mask_sem_marcacao.sum() < n_sem_marcacao_min:
#     raise ValueError(
#         f"Não há registros suficientes sem marcação. Disponíveis: {mask_sem_marcacao.sum()}."
#     )

# selected_idx = set()

# # 1) Garante pelo menos 15 sem marcação
# idx_sem = df_estabelecimentos.loc[mask_sem_marcacao].sample(
#     n=n_sem_marcacao_min, random_state=seed
# ).index
# selected_idx.update(idx_sem)

# # 2) Garante cobertura de todas as regras não zeradas
# for i, regra in enumerate(regras_nao_zeradas, start=1):
#     # Se já existe algum selecionado com essa regra, segue
#     if selected_idx and df_estabelecimentos.loc[list(selected_idx), regra].sum() > 0:
#         continue

#     candidatos = df_estabelecimentos.index[df_estabelecimentos[regra] > 0]
#     candidatos_nao_sel = candidatos.difference(pd.Index(selected_idx))

#     if len(candidatos_nao_sel) > 0:
#         idx = df_estabelecimentos.loc[candidatos_nao_sel].sample(
#             n=1, random_state=seed + i
#         ).index[0]
#     else:
#         # fallback (caso extremo: só exista registro já selecionado)
#         idx = df_estabelecimentos.loc[candidatos].sample(
#             n=1, random_state=seed + i
#         ).index[0]

#     selected_idx.add(idx)

# if len(selected_idx) > n_total:
#     raise ValueError(
#         f"Quantidade mínima obrigatória ({len(selected_idx)}) excede o total desejado ({n_total})."
#     )

# # 3) Completa aleatoriamente até 60
# faltam = n_total - len(selected_idx)
# if faltam > 0:
#     restantes = df_estabelecimentos.index.difference(pd.Index(selected_idx))
#     idx_extra = df_estabelecimentos.loc[restantes].sample(
#         n=faltam, random_state=seed + 999
#     ).index
#     selected_idx.update(idx_extra)

# # DataFrame final de amostra
# df_sample_60 = (
#     df_estabelecimentos.loc[list(selected_idx)]
#     .sample(frac=1, random_state=seed)
#     .reset_index(drop=True)
# )

# # Validações
# assert len(df_sample_60) == n_total
# assert (df_sample_60[colunas_regras].sum(axis=1) == 0).sum() >= n_sem_marcacao_min
# assert (df_sample_60[regras_nao_zeradas].sum(axis=0) > 0).all()

# print(f"Amostra final: {len(df_sample_60)} registros")
# print(f"Sem marcação: {(df_sample_60[colunas_regras].sum(axis=1) == 0).sum()}")
# print(f"Regras não zeradas cobertas: {(df_sample_60[regras_nao_zeradas].sum(axis=0) > 0).sum()}/{len(regras_nao_zeradas)}")

# # Opcional: visualizar chaves + regras
# display(df_sample_60[key_cols + colunas_regras].head())

In [ ]:
# # Cria um novo SQLite somente com os formulários da amostra (df_sample_60[key_cols])
# source_db = DB_MAIN if "DB_MAIN" in globals() else "../data/amstr_geral_criticas.db"
# dest_db = "../data/amstr_geral_sample_60.db"

# keys_amostra = df_sample_60[key_cols].drop_duplicates().copy()

# with sqlite3.connect(source_db) as conn:
#     conn.execute(f"ATTACH DATABASE '{dest_db}' AS out")

#     # Limpa tabelas existentes no destino
#     tabelas_out = [
#         r[0]
#         for r in conn.execute(
#             "SELECT name FROM out.sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';"
#         ).fetchall()
#     ]
#     for tabela in tabelas_out:
#         conn.execute(f'DROP TABLE out."{tabela}"')

#     # Tabela temporária com as chaves selecionadas
#     keys_amostra.to_sql("_tmp_keys_sample60", conn, if_exists="replace", index=False)
#     conn.execute(
#         f'CREATE INDEX IF NOT EXISTS idx_tmp_keys_sample60 ON _tmp_keys_sample60 ({", ".join(key_cols)})'
#     )

#     tabelas_main = [
#         r[0]
#         for r in conn.execute(
#             "SELECT name FROM main.sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';"
#         ).fetchall()
#     ]

#     resumo = []
#     tabelas_sem_chave = []

#     for tabela in tabelas_main:
#         cols_tabela = [r[1] for r in conn.execute(f'PRAGMA main.table_info("{tabela}")').fetchall()]

#         if all(c in cols_tabela for c in key_cols):
#             join_cond = " AND ".join([f't."{c}" = k."{c}"' for c in key_cols])
#             conn.execute(
#                 f"""
#                 CREATE TABLE out."{tabela}" AS
#                 SELECT t.*
#                 FROM main."{tabela}" t
#                 INNER JOIN _tmp_keys_sample60 k
#                   ON {join_cond}
#                 """
#             )
#             qtd = conn.execute(f'SELECT COUNT(*) FROM out."{tabela}"').fetchone()[0]
#             resumo.append((tabela, qtd))
#         else:
#             tabelas_sem_chave.append(tabela)

#     conn.execute("DROP TABLE IF EXISTS _tmp_keys_sample60")
#     conn.commit()
#     conn.execute("DETACH DATABASE out")

# resumo_copia = (
#     pd.DataFrame(resumo, columns=["tabela", "qtd_registros"])
#     .sort_values("qtd_registros", ascending=False)
#     .reset_index(drop=True)
# )

# print(f"Arquivo criado: {dest_db}")
# print(f"Tabelas copiadas: {len(resumo_copia)}")
# print(f"Tabelas ignoradas (sem key_cols): {len(tabelas_sem_chave)}")
# display(resumo_copia)